In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv("/Users/dannyharlan/Desktop/Copy of HopefullyFixed - Sheet1.csv")


# Extract year from parentheses into a new column before removing it
df["Year"] = df["Name"].str.extract(r"\((\d+)\)")  # Extracts the numeric part inside parentheses

# Now remove the year in parentheses from 'Name' column
df["Name"] = df["Name"].str.replace(r"\s*\(\d+\)", "", regex=True).str.strip()

# Preview the first few rows
print(df.head())

# Optional: Check data types and convert Year to numeric if needed
print("\nData types before conversion:")
print(df.dtypes)

df["Year"] = pd.to_numeric(df["Year"], errors='coerce')  # Convert to numeric, invalid parsing becomes NaN

print("\nData types after conversion:")
print(df.dtypes)

# Remove rows where Year is 8 or 9
df = df[~df['Year'].isin([8, 9])]

# Alternative syntax
df = df[(df['Year'] != 8) & (df['Year'] != 9)]
df.head(10)




In [ ]:
import pandas as pd

# Load the data
cluster_df = pd.read_csv("/Users/dannyharlan/Desktop/FinalCleaned.csv")


# Extract year from parentheses into a new column before removing it
cluster_df["Year"] = cluster_df["Name"].str.extract(r"\((\d+)\)")  # Extracts the numeric part inside parentheses

# Now remove the year in parentheses from 'Name' column
cluster_df["Name"] = cluster_df["Name"].str.replace(r"\s*\(\d+\)", "", regex=True).str.strip()

# Preview the first few rows
print(cluster_df.head())

# Optional: Check data types and convert Year to numeric if needed
print("\nData types before conversion:")
print(cluster_df.dtypes)

cluster_df["Year"] = pd.to_numeric(df["Year"], errors='coerce')  # Convert to numeric, invalid parsing becomes NaN

print("\nData types after conversion:")
print(cluster_df.dtypes)

# Remove rows where Year is 8 or 9
cluster_df = cluster_df[~cluster_df['Year'].isin([8, 9])]

# Alternative syntax
cluster_df = cluster_df[(cluster_df['Year'] != 8) & (cluster_df['Year'] != 9)]
df.head(10)


In [ ]:
# Define class rank order
class_order = {"Fr": 1, "So": 2, "Jr": 3, "Sr": 4}

# Map class year to numeric rank
df["class_rank"] = df["Player"].map(class_order)

# Sort so older classes (higher rank) come first
df = df.sort_values(by=["Name", "class_rank", "Height"], ascending=[True, False, True])

# Drop duplicates based on both Name + Height
df = df.drop_duplicates(subset=["Name"], keep="first")


# Drop helper column
df = df.drop(columns="class_rank")








In [ ]:
df.head(10)

In [ ]:
times_df = pd.read_csv("/Users/dannyharlan/Desktop/CombineTimes - Sheet1.csv")
measures_df = pd.read_csv("/Users/dannyharlan/Desktop/Combine Measurables - Sheet1.csv")
year_df = pd.read_csv("/Users/dannyharlan/Desktop/YearFix - Sheet1.csv")

In [ ]:
# Count how many times each name appears
name_counts = df["Name"].value_counts()

# Filter to names that appear more than once
duplicate_names = name_counts[name_counts > 1]

# Display them
print(duplicate_names)


In [ ]:
measures_df.head(10)

In [ ]:
print("times_df columns:", times_df.columns.tolist())
print("measures_df columns:", measures_df.columns.tolist())


In [ ]:
# Step 1: Keep only the first occurrence for each player in times_df and measures_df
times_df_unique = times_df.drop_duplicates(subset="PLAYER", keep="first")
measures_df_unique = measures_df.drop_duplicates(subset="PLAYER", keep="first")

# Step 2: Merge with df using left join on Name
merged_df = df.merge(times_df_unique, how="left", left_on="Name", right_on="PLAYER")
merged_df = merged_df.merge(measures_df_unique, how="left", left_on="Name", right_on="PLAYER")

# Step 3: Drop the extra 'player' columns added during merge
merged_df = merged_df.drop(columns=["player_x", "player_y"], errors="ignore")  # or just "player" if it's not duplicated


In [ ]:
merged_df.head(10)

In [ ]:
merged_df.columns

In [ ]:
advanced_nba_df = pd.read_csv("/Users/dannyharlan/Downloads/PlayerStats.xlsx - Sheet1.csv")

In [ ]:
advanced_nba_df.columns


In [ ]:
# Make sure player name is clean
advanced_nba_df["player_clean"] = advanced_nba_df["Player"].str.split("\\").str[0].str.strip()

# For each player, keep only their first 4 rows (chronologically sorted)
first_4_seasons = advanced_nba_df.groupby("player_clean").head(4)

# Sum VORP and BPM for first 4 seasons
vorp_bpm_sums = first_4_seasons.groupby("player_clean")[["VORP", "BPM"]].sum().reset_index()

# Optional: rename columns for clarity
vorp_bpm_sums = vorp_bpm_sums.rename(columns={"VORP": "sum_vorp_4yrs", "BPM": "sum_bpm_4yrs"})


In [ ]:
vorp_bpm_sums.head(10)

In [ ]:
# Merge the VORP/BPM sums with the main dataframe
final_df = merged_df.merge(
    vorp_bpm_sums,
    how="left",
    left_on="Name",
    right_on="player_clean"
)

# Drop the redundant player_clean column if you want
final_df = final_df.drop(columns=["player_clean"], errors="ignore")

# Preview the final merged dataframe
final_df.head(10)

In [ ]:
import numpy as np
# Slightly curved: linear * gentle inverse
final_df["DraftValue"] = (1 - final_df["Pick"] / 60) * (1 / (final_df["Pick"] ** 0.1))







In [ ]:
# Count occurrences of each name
name_counts = final_df['Name'].value_counts()
duplicate_name_counts = name_counts[name_counts > 1]
print("Names that appear more than once:")
print(duplicate_name_counts)

In [ ]:
# Optional: clean casing/spacing if still mismatches
final_df["Name_clean"] = final_df["Name"].str.lower().str.strip()
year_df["Name_clean"] = year_df["Player"].str.lower().str.strip()

# Recheck match rate
name_matches = final_df["Name_clean"].isin(year_df["Name_clean"])
print(f"Matched names: {name_matches.sum()} / {len(final_df)}")


In [ ]:
unmatched_names = final_df[~final_df["Name_clean"].isin(year_df["Name_clean"])]["Name"].unique()
print(f"Unmatched names ({len(unmatched_names)}):")
print(unmatched_names)


In [ ]:
print(year_df["Year"].value_counts().sort_index())

In [ ]:
year_df.head()

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)  # Show all columns
pd.set_option("display.width", 0)           # Don't break lines

print(final_df.head())  # or .sample(5) for random rows


In [ ]:
# Check how well names overlap
name_matches = final_df["Name"].isin(year_df["Player"])
print(f"Matched names: {name_matches.sum()} / {len(final_df)}")


In [ ]:
final_df.columns
final_df["Ast/TO"] = final_df["Ast"] / final_df["TO"]


In [ ]:
# Encode Player column in final_df_filtered
# Fr: 1, So: 2, Jr: 3, Sr: 4

# Create the encoding mapping
player_encoding = {
    'Fr': 1,
    'So': 2,
    'Jr': 3,
    'Sr': 4
}

# Apply the encoding
if 'Player' in final_df.columns:
    # Show original distribution
    print("Original Player column distribution:")
    print(final_df['Player'].value_counts())

    # Create encoded column
    final_df['Player_Encoded'] = final_df['Player'].map(player_encoding)

    # Show encoded distribution
    print("\nEncoded Player column distribution:")
    print(final_df['Player_Encoded'].value_counts().sort_index())

    # Check for any missing values (in case there were unexpected class years)
    missing_encoded = final_df['Player_Encoded'].isna().sum()
    if missing_encoded > 0:
        print(f"\nWarning: {missing_encoded} rows could not be encoded")
        print("Unique values in Player column:", final_df['Player'].unique())

    print(f"\nSuccessfully encoded Player column!")
    print(f"New column 'Player_Encoded' created with values 1-4")

else:
    print("Player column not found in final_df_filtered")
    print("Available columns:", list(final_df.columns))

In [ ]:
# Sort by sum_vorp_4yrs in descending order and display top results
top_vorp_players = final_df.sort_values('sum_vorp_4yrs', ascending=False)[['Name', 'sum_vorp_4yrs']]

# Display top 20 players by VORP
print("Top 20 Players by 4-Year VORP Sum:")
print(top_vorp_players.head(20).to_string(index=False))

# Optional: Visualize the top players
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
top_vorp_players.head(10).plot.bar(x='Name', y='sum_vorp_4yrs', color='skyblue')
plt.title('Top 10 Players by 4-Year VORP Sum')
plt.ylabel('Total VORP (first 4 seasons)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Calculate correlation matrix for numeric columns
corr_matrix = final_df.select_dtypes(include=['number']).corr()

# Show correlations with sum_vorp_4yrs, sorted by absolute value
vorp_correlations = corr_matrix['sum_vorp_4yrs'].sort_values(key=abs, ascending=False)

print("Correlation with sum_vorp_4yrs (numeric features):")
print(vorp_correlations.to_string())

In [ ]:
# For categorical columns (if any), you can analyze group means
categorical_cols = final_df.select_dtypes(include=['object', 'category']).columns

for col in categorical_cols:
    if col != 'Name':  # Skip player names
        print(f"\nAverage sum_vorp_4yrs by {col}:")
        print(final_df.groupby(col)['sum_vorp_4yrs'].mean().sort_values(ascending=False).to_string())

In [ ]:
# Split into makes and attempts
final_df[['close_makes', 'close_attempts']] = final_df['Close 2 Raw'].str.split('-', expand=True).astype(float)

# Calculate additional meaningful metrics
final_df['close_pct'] = final_df['close_makes'] / final_df['close_attempts']
final_df['close_missed'] = final_df['close_attempts'] - final_df['close_makes']
final_df

In [ ]:
# Keep the raw components
final_df['close_makes'] = final_df['Close 2 Raw'].str.split('-').str[0].astype(float)
final_df['close_attempts'] = final_df['Close 2 Raw'].str.split('-').str[1].astype(float)

# Create interaction terms
final_df['close_volume_x_pct'] = final_df['close_attempts'] * (final_df['close_makes'] / final_df['close_attempts'])
final_df["close_quality_volume"] = ((final_df["close_makes"] * 2) ** 2) * (final_df["close_makes"] / final_df["close_attempts"])
final_df['close_efficiency_dominance'] = (final_df['close_pct'] ** 4) * (final_df['close_attempts'] ** 0.5)


# Basketball-specific metrics
final_df['close_pts_per_100'] = (final_df['close_makes'] * 2) / final_df['close_attempts'] * 100  # Points per 100 attempts

In [ ]:
# Split into makes and attempts
final_df[['far_makes', 'far_attempts']] = final_df['Far Two Raw'].str.split('-', expand=True).astype(float)

# Calculate additional meaningful metrics
final_df['far_pct'] = final_df['far_makes'] / final_df['far_attempts']
final_df['far_missed'] = final_df['far_attempts'] - final_df['far_makes']

In [ ]:
# Keep the raw components
final_df['far_makes'] = final_df['Far Two Raw'].str.split('-').str[0].astype(float)
final_df['far_attempts'] = final_df['Far Two Raw'].str.split('-').str[1].astype(float)

# Create interaction terms
final_df['far_volume_x_pct'] = final_df['far_attempts'] * (final_df['far_makes'] / final_df['far_attempts'])
final_df['far_quality_volume'] = (final_df['far_makes']**2) / final_df['far_attempts']  # Favors efficient volume

# Basketball-specific metrics
final_df['far_pts_per_100'] = (final_df['far_makes'] * 2) / final_df['far_attempts'] * 100  # Points per 100 attempts

In [ ]:
# Split into makes and attempts
final_df[['three_makes', 'three_attempts']] = final_df['3P Raw'].str.split('-', expand=True).astype(float)
final_df['Close 2 %'] = final_df['Close 2 %'].clip(upper=0.65)

# Calculate additional meaningful metrics
final_df['three_pct'] = final_df['three_makes'] / final_df['three_attempts']
final_df['three_missed'] = final_df['three_attempts'] - final_df['three_makes']

In [ ]:
# Keep the raw components
final_df['three_makes'] = final_df['3P Raw'].str.split('-').str[0].astype(float)
final_df['three_attempts'] = final_df['3P Raw'].str.split('-').str[1].astype(float)

# Create interaction terms
final_df['three_volume_x_pct'] = final_df['three_attempts'] * (final_df['three_makes'] / final_df['three_attempts'])
final_df['three_quality_volume'] = (final_df['three_makes']**2) / final_df['three_attempts']  # Favors efficient volume

# Basketball-specific metrics
final_df['three_pts_per_100'] = (final_df['three_makes'] * 2) / final_df['three_attempts'] * 100  # Points per 100



In [ ]:
# Method 1: Using isin() - most straightforward
power_conferences = ['B10', 'SEC', 'ACC', 'P12', 'BE', 'B12']
final_df['Power_Conf'] = final_df['Conf'].isin(power_conferences).astype(int)

In [ ]:
final_df.head(10)

In [ ]:
def height_to_inches(height_str):
    if pd.isna(height_str):
        return None
    try:
        feet, inches = height_str.split("-")
        return int(feet) * 12 + int(inches)
    except:
        return None

# Apply conversion
final_df["Height_in"] = final_df["Height"].apply(height_to_inches)


In [ ]:
def assign_cluster_by_role(role, height_in, offensive_rebounds, defensive_rebounds, three_point_per_100):
  # Determine role-based cluster first
  if pd.isna(role):
      role_cluster = 2.0  # Default to guards if no role specified
  else:
      role_str = str(role).strip()

      # Cluster 0.0: Centers and Power Forward/Centers
      if role_str in ['C', 'PF/C']:
          role_cluster = 0.0

      # Cluster 1.0: Stretch 4s and Wing Forwards
      elif role_str in ['Stretch 4', 'Wing F']:
          role_cluster = 1.0

      # Everything else would normally be 2.0 (Guards)
      else:
          role_cluster = 2.0

  # Track if originally wings
  originally_wings = (role_cluster == 1.0)

  # FIRST PRIORITY: OR adjustment - if OR < 3.3, force to guards
  if not pd.isna(offensive_rebounds) and offensive_rebounds < 3.3:
      role_cluster = 2.0

  # SECOND PRIORITY: DR adjustment - if DR >= 27, force to centers
  elif not pd.isna(defensive_rebounds) and defensive_rebounds >= 27:
      return 0.0
  elif not pd.isna(defensive_rebounds) and defensive_rebounds <= 10.9:
      role_cluster = 2.0

  # Height adjustment: if they would be 2.0, height >= 77", AND have good rebounding, make them 1.0
  # Don't apply if originally wings (don't override rebounding demotion)
  if role_cluster == 2.0 and not originally_wings and not pd.isna(height_in) and height_in >= 77:
      # Only promote if they have good enough rebounding
      has_good_rebounding = (not pd.isna(offensive_rebounds) and offensive_rebounds >= 3.3) and \
                            (not pd.isna(defensive_rebounds) and defensive_rebounds > 10.9)
      if has_good_rebounding:
          return 1.0

  # 3P adjustment: if originally wings, now 2.0, and 3P/100 <= 5, move back to 1.0
  if originally_wings and role_cluster == 2.0 and not pd.isna(three_point_per_100) and three_point_per_100 <= 5:
      return 1.0

  return role_cluster

# Apply clustering with the new parameters
final_df['PlayStyleCluster'] = final_df.apply(lambda row: assign_cluster_by_role(row['Role'], row['Height_in'], row['OR'], row['DR'], row['3P/100']), axis=1)

# Check how many players were affected by the new DR >= 27 rule
dr_high_players = final_df[(final_df['DR'] >= 27) & (final_df['OR'] >= 3.3)]
print(f"Players with DR >= 27 (not already forced to guards by OR < 3.3): {len(dr_high_players)}")
print(f"These players are now assigned to cluster 0.0 (Centers)")

# Check how many players moved back to 1.0 by 3P rule
wings_back_by_3p = final_df[(final_df['Role'].isin(['Stretch 4', 'Wing F'])) & (final_df['PlayStyleCluster'] == 1.0) & (final_df['3P/100'] <= 5) &
((final_df['OR'] < 3.3) | (final_df['DR'] <= 10.9))]
print(f"\nOriginal wings moved back to 1.0 by 3P/100 <= 5: {len(wings_back_by_3p)}")

print("\nFinal cluster distribution:")
print(final_df['PlayStyleCluster'].value_counts().sort_index())



In [ ]:
def height_to_inches(height_str):
    if pd.isna(height_str):
        return None
    try:
        feet, inches = height_str.split("-")
        return int(feet) * 12 + int(inches)
    except:
        return None

# Apply conversion
cluster_df["Height_in"] = cluster_df["Height"].apply(height_to_inches)

In [ ]:
import re

def wingspan_to_inches(w):
    if pd.isna(w):
        return None
    # Match formats like 7' 5.5'' or 6'11''
    match = re.match(r"(\d+)'[\s]*(\d+(?:\.\d*)?)?\"?", w)
    if match:
        feet = int(match.group(1))
        inches = float(match.group(2)) if match.group(2) else 0
        return feet * 12 + inches
    return None

# Apply to your Wingspan column (replace 'WINGSPAN' with your exact column name)
final_df["Wingspan_in"] = final_df["WINGSPAN"].apply(wingspan_to_inches)

# Substitute empty Wingspan_in with Height_in
print("Before substitution:")
print(f"Missing Wingspan_in values: {final_df['Wingspan_in'].isna().sum()}")
print(f"Missing Height_in values: {final_df['Height_in'].isna().sum()}")


In [ ]:

final_df['Wingspan_in'] = final_df['Wingspan_in'].fillna(final_df['Height_in']) +3
final_df["Wing_HeightDiff"] =  final_df["Wingspan_in"] - final_df["Height_in"]

In [ ]:
final_df['Usg/Ast'] = final_df["Usg"] / final_df["Ast"]
final_df['Usg/Ast/TO'] = final_df["Usg"] / final_df["Ast/TO"]
final_df['Usg/TO'] = final_df["Usg"] / final_df["TO"]

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Choose features you used to cluster
features = ["Height_in", "Blk", 'Ast']
X = final_df[features].dropna()
X_scaled = (X - X.mean()) / X.std()

# Perform PCA
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

# Plot
plt.figure(figsize=(8, 6))
plt.scatter(components[:, 0], components[:, 1], c=final_df.loc[X.index, "PlayStyleCluster"], cmap="tab10", s=50)
plt.title("Player Style Clusters (PCA)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.colorbar(label="Cluster")
plt.show()


In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_components = tsne.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(tsne_components[:, 0], tsne_components[:, 1], c=final_df.loc[X.index, "PlayStyleCluster"], cmap="tab10", s=50)
plt.title("Player Style Clusters (t-SNE)")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.grid(True)
plt.colorbar(label="Cluster")
plt.show()


In [ ]:
final_df.columns

In [ ]:
advanced_nba_df.columns

In [ ]:
# Find each player's first VORP year
player_first_vorp_year = advanced_nba_df.groupby('player_clean')['Year'].min()

# Get players whose first VORP was in 2018 or earlier
players_started_by_2018 = player_first_vorp_year[player_first_vorp_year <= 2018].index

# Filter final_df to only include those players
final_df_filtered = final_df[final_df['Name'].isin(players_started_by_2018)]

# Normalize target within each cluster
final_df_filtered["zscore_vorp"] = np.nan

for cluster_id in final_df_filtered["PlayStyleCluster"].dropna().unique():
    mask = final_df_filtered["PlayStyleCluster"] == cluster_id
    vorp = final_df_filtered.loc[mask, "sum_vorp_4yrs"]
    final_df_filtered.loc[mask, "zscore_vorp"] = (vorp - vorp.mean()) / vorp.std()

final_df["Ast/TO"] = final_df["Ast"] / final_df["TO"]
final_df_filtered["Ast/TO"] = final_df_filtered["Ast"] / final_df_filtered["TO"]


print(f"Players who started by 2018: {len(players_started_by_2018)}")
print(f"Original final_df: {len(final_df)} rows")
print(f"Filtered final_df: {len(final_df_filtered)} rows")
print(f"Removed {len(final_df) - len(final_df_filtered)} rows")

In [ ]:
final_df.head(10)

In [ ]:
final_df.columns

In [ ]:
final_df['REB'] = final_df['OR'] + final_df['DR']
final_df.head(10)

In [ ]:
# Calculate correlation matrix for numeric columns
corr_matrix = final_df.select_dtypes(include=['number']).corr()

# Show correlations with sum_vorp_4yrs, sorted by absolute value
vorp_correlations = corr_matrix['sum_vorp_4yrs'].sort_values(key=abs, ascending=False)

print("Correlation with sum_vorp_4yrs (numeric features):")
print(vorp_correlations.to_string())

In [ ]:
final_df_filtered['REB'] = final_df_filtered["OR"] + final_df_filtered["DR"]

In [ ]:
import pandas as pd
import numpy as np

def log_transform_features(df, feature_list, method='log1p'):
    """
    Apply log transformation to multiple features and create new columns.

    Parameters:
    df: pandas DataFrame
    feature_list: list of column names to transform
    method: 'log1p' (default), 'log', or 'log_shift'
        - 'log1p': log(1 + x) - handles zeros
        - 'log': natural log - requires all positive values
        - 'log_shift': log(x - min(x) + 1) - handles negative values

    Returns:
    DataFrame with new log-transformed columns
    """

    # Make a copy to avoid modifying original
    df_new = df.copy()

    for feature in feature_list:
        if feature not in df.columns:
            print(f"Warning: '{feature}' not found in DataFrame columns")
            continue

        new_col_name = f"Log{feature}"

        try:
            if method == 'log1p':
                # log(1 + x) - good for data with zeros
                df_new[new_col_name] = np.log1p(df[feature])

            elif method == 'log':
                # Natural log - requires all positive values
                if (df[feature] <= 0).any():
                    print(f"Warning: '{feature}' contains zeros or negative values. Using log1p instead.")
                    df_new[new_col_name] = np.log1p(df[feature])
                else:
                    df_new[new_col_name] = np.log(df[feature])

            elif method == 'log_shift':
                # Shift to handle negative values
                min_val = df[feature].min()
                if min_val <= 0:
                    df_new[new_col_name] = np.log(df[feature] - min_val + 1)
                else:
                    df_new[new_col_name] = np.log(df[feature])

            print(f"✓ Created '{new_col_name}' from '{feature}'")

        except Exception as e:
            print(f"Error transforming '{feature}': {e}")

    return df_new

# Apply log transformation to your specific features
features_to_transform = ['Ast', 'REB', 'DR', 'OR', 'Ast/TO', 'Blk', 'Stl', 'Usg/Ast', 'FT%', 'FTR', 'close_volume_x_pct', 'Usg/Ast/TO', 'Usg', 'three_volume_x_pct', 'BPM', 'DBPM', 'OBPM', 'close_quality_volume', 'close_makes', '3P/100', 'Close 2 %', 'TS']

# Transform the features
final_df_transformed = log_transform_features(final_df_filtered, features_to_transform)

# Display the results - show original vs transformed
print("Original vs Log-transformed columns:")
for feature in features_to_transform:
    if feature in final_df_filtered.columns:
        log_feature = f"Log{feature}"
        print(f"\n{feature} -> {log_feature}")
        print(f"Original range: {final_df_filtered[feature].min():.2f} to {final_df_filtered[feature].max():.2f}")
        if log_feature in final_df_transformed.columns:
            print(f"Log range: {final_df_transformed[log_feature].min():.2f} to {final_df_transformed[log_feature].max():.2f}")

# Show first few rows of original vs transformed
print("\nFirst 5 rows comparison:")
cols_to_show = []
for feature in features_to_transform:
    if feature in final_df_filtered.columns:
        cols_to_show.extend([feature, f"Log{feature}"])

if cols_to_show:
    print(final_df_transformed[cols_to_show].head())

print(f"\nDataFrame shape: {final_df_transformed.shape}")
print(f"New log-transformed columns added: {len([col for col in final_df_transformed.columns if col.startswith('Log')])}")

In [ ]:
import pandas as pd
import numpy as np

def log_transform_features(df, feature_list, method='log1p'):
    """
    Apply log transformation to multiple features and create new columns.

    Parameters:
    df: pandas DataFrame
    feature_list: list of column names to transform
    method: 'log1p' (default), 'log', or 'log_shift'
        - 'log1p': log(1 + x) - handles zeros
        - 'log': natural log - requires all positive values
        - 'log_shift': log(x - min(x) + 1) - handles negative values

    Returns:
    DataFrame with new log-transformed columns
    """

    # Make a copy to avoid modifying original
    df_new = df.copy()

    for feature in feature_list:
        if feature not in df.columns:
            print(f"Warning: '{feature}' not found in DataFrame columns")
            continue

        new_col_name = f"Log{feature}"

        try:
            if method == 'log1p':
                # log(1 + x) - good for data with zeros
                df_new[new_col_name] = np.log1p(df[feature])

            elif method == 'log':
                # Natural log - requires all positive values
                if (df[feature] <= 0).any():
                    print(f"Warning: '{feature}' contains zeros or negative values. Using log1p instead.")
                    df_new[new_col_name] = np.log1p(df[feature])
                else:
                    df_new[new_col_name] = np.log(df[feature])

            elif method == 'log_shift':
                # Shift to handle negative values
                min_val = df[feature].min()
                if min_val <= 0:
                    df_new[new_col_name] = np.log(df[feature] - min_val + 1)
                else:
                    df_new[new_col_name] = np.log(df[feature])

            print(f"✓ Created '{new_col_name}' from '{feature}'")

        except Exception as e:
            print(f"Error transforming '{feature}': {e}")

    return df_new

# Apply log transformation to your specific features
features_to_transform = ['Ast', 'REB', 'DR', 'OR', 'Ast/TO', 'Blk', 'Stl', 'Usg/Ast', 'FT%', 'FTR', 'close_volume_x_pct', 'Usg/Ast/TO', 'Usg', 'three_volume_x_pct', 'close_quality_volume', 'close_makes', '3P/100', 'Close 2 %', 'TS', 'DBPM']

# Transform the features
final_df_transform = log_transform_features(final_df, features_to_transform)

# Display the results - show original vs transformed
print("Original vs Log-transformed columns:")
for feature in features_to_transform:
    if feature in final_df.columns:
        log_feature = f"Log{feature}"
        print(f"\n{feature} -> {log_feature}")
        print(f"Original range: {final_df[feature].min():.2f} to {final_df[feature].max():.2f}")
        if log_feature in final_df_transform.columns:
            print(f"Log range: {final_df_transform[log_feature].min():.2f} to {final_df_transform[log_feature].max():.2f}")

# Show first few rows of original vs transformed
print("\nFirst 5 rows comparison:")
cols_to_show = []
for feature in features_to_transform:
    if feature in final_df.columns:
        cols_to_show.extend([feature, f"Log{feature}"])

if cols_to_show:
    print(final_df_transform[cols_to_show].head())

print(f"\nDataFrame shape: {final_df_transform.shape}")
print(f"New log-transformed columns added: {len([col for col in final_df_transformed.columns if col.startswith('Log')])}")

In [ ]:
final_df_transform[["LogLogDR", "LogLogREB"]] = np.log1p(final_df_transform[["LogDR", "LogREB"]])
final_df_transformed[["LogLogDR", "LogLogREB"]] = np.log1p(final_df_transformed[["LogDR", "LogREB"]])

In [ ]:
# Calculate 90th percentile from the unlogged 'DR' column
dr_90 = final_df_transformed["DR"].quantile(0.90)

# Clip DR in both DataFrames
final_df_transformed["DR_clipped"] = final_df_transformed["DR"].clip(upper=dr_90)
final_df_transform["DR_clipped"] = final_df_transform["DR"].clip(upper=dr_90)

# Log-transform the clipped DR
final_df_transformed["LogDR_clipped"] = np.log1p(final_df_transformed["DR_clipped"])
final_df_transform["LogDR_clipped"] = np.log1p(final_df_transform["DR_clipped"])





In [ ]:
# Calculate 90th percentile from the unlogged 'DR' column
REB_90 = final_df_transformed["REB"].quantile(0.97)

# Clip DR in both DataFrames
final_df_transformed["REB_clipped"] = final_df_transformed["REB"].clip(upper=dr_90)
final_df_transform["REB_clipped"] = final_df_transform["REB"].clip(upper=dr_90)

# Log-transform the clipped DR
final_df_transformed["LogREB_clipped"] = np.log1p(final_df_transformed["REB_clipped"])
final_df_transform["LogREB_clipped"] = np.log1p(final_df_transform["REB_clipped"])
final_df_transform["close_min"]  = final_df_transform['close_makes'] / final_df_transform['Min%']



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Filter data for PlayStyleCluster = 0.0
cluster_0_data = final_df_transformed[final_df_transformed['PlayStyleCluster'] == 0.0].copy()

print(f"Filtered data shape: {cluster_0_data.shape}")

# Check data types
print(f"\nData types in dataset:")
print(cluster_0_data.dtypes.value_counts())

# Identify numeric columns only
numeric_cols = cluster_0_data.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns found: {len(numeric_cols)}")

# Check if target exists and is numeric
if 'sum_vorp_4yrs' not in numeric_cols:
    print("ERROR: sum_vorp_4yrs is not numeric or not found!")
    print("Available columns:", list(cluster_0_data.columns))
    print("sum_vorp_4yrs dtype:", cluster_0_data['sum_vorp_4yrs'].dtype if 'sum_vorp_4yrs' in cluster_0_data.columns else "Column not found")
else:
    print(f"Target variable (sum_vorp_4yrs) distribution:")
    print(cluster_0_data['sum_vorp_4yrs'].describe())

# Create numeric-only subset for correlations
numeric_data = cluster_0_data[numeric_cols]

# Calculate correlations with sum_vorp_4yrs (only if target is numeric)
if 'sum_vorp_4yrs' in numeric_cols:
    correlations = numeric_data.corr()['sum_vorp_4yrs'].sort_values(ascending=False)
else:
    print("Cannot calculate correlations - target variable not numeric")
    correlations = pd.Series(dtype=float)

if len(correlations) > 0:
    print("\n=== CORRELATIONS WITH sum_vorp_4yrs ===")
    print("Top positive correlations:")
    print(correlations.head(15))

    print("\nTop negative correlations:")
    print(correlations.tail(15))

    # Remove the target variable itself and any NaN correlations
    correlations_clean = correlations.drop('sum_vorp_4yrs').dropna()

    print(f"\n=== STRONGEST CORRELATIONS (excluding target) ===")
    print("Strongest absolute correlations:")
    correlations_abs = correlations_clean.abs().sort_values(ascending=False)
    print(correlations_abs.head(20))
else:
    print("\n=== NO CORRELATIONS AVAILABLE ===")
    print("Target variable may not be numeric or no numeric columns found")
    correlations_clean = pd.Series(dtype=float)
    correlations_abs = pd.Series(dtype=float)

# Create correlation matrix for visualization (numeric columns only)
corr_matrix = numeric_data.corr()

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

# 1. Target distribution
if 'sum_vorp_4yrs' in numeric_cols:
    axes[0,0].hist(cluster_0_data['sum_vorp_4yrs'], bins=20, alpha=0.7, edgecolor='black')
    axes[0,0].set_title('Distribution of sum_vorp_4yrs')
    axes[0,0].set_xlabel('sum_vorp_4yrs')
    axes[0,0].set_ylabel('Frequency')
else:
    axes[0,0].text(0.5, 0.5, 'sum_vorp_4yrs\nnot numeric',
                  ha='center', va='center', transform=axes[0,0].transAxes)
    axes[0,0].set_title('Target Variable Issue')

# 2. Correlation bar plot
top_corrs = correlations_clean.abs().sort_values(ascending=False).head(15)
top_corrs_signed = correlations_clean[top_corrs.index]

colors = ['red' if x < 0 else 'blue' for x in top_corrs_signed.values]
axes[0,1].barh(range(len(top_corrs_signed)), top_corrs_signed.values, color=colors, alpha=0.7)
axes[0,1].set_yticks(range(len(top_corrs_signed)))
axes[0,1].set_yticklabels(top_corrs_signed.index, fontsize=8)
axes[0,1].set_xlabel('Correlation with sum_vorp_4yrs')
axes[0,1].set_title('Top 15 Correlations with sum_vorp_4yrs')
axes[0,1].axvline(x=0, color='black', linestyle='-', alpha=0.3)

# 3. Correlation heatmap (top correlated features only)
if len(correlations_abs) > 0:
    # Get top 20 most correlated features + target
    top_features = ['sum_vorp_4yrs'] + list(correlations_abs.head(19).index)
    # Make sure all features exist in the data
    top_features = [f for f in top_features if f in cluster_0_data.columns]

    if len(top_features) > 1:
        subset_corr = cluster_0_data[top_features].corr()
        mask = np.triu(np.ones_like(subset_corr, dtype=bool))

        sns.heatmap(subset_corr, mask=mask, annot=True, cmap='RdBu_r', center=0,
                   square=True, ax=axes[1,0], fmt='.2f', cbar_kws={'shrink': 0.8})
        axes[1,0].set_title('Correlation Matrix (Top Features)')
    else:
        axes[1,0].text(0.5, 0.5, 'Not enough features\nfor correlation matrix',
                      ha='center', va='center', transform=axes[1,0].transAxes)

# 4. Scatter plot of strongest correlation
if len(correlations_abs) > 0:
    strongest_feature = correlations_abs.index[0]
    strongest_corr = correlations_clean[strongest_feature]

    axes[1,1].scatter(cluster_0_data[strongest_feature], cluster_0_data['sum_vorp_4yrs'],
                     alpha=0.6, s=50)
    axes[1,1].set_xlabel(strongest_feature)
    axes[1,1].set_ylabel('sum_vorp_4yrs')
    axes[1,1].set_title(f'Strongest Correlation: {strongest_feature}\nr = {strongest_corr:.3f}')

    # Add trend line
    if not cluster_0_data[strongest_feature].isna().all():
        z = np.polyfit(cluster_0_data[strongest_feature].dropna(),
                      cluster_0_data.loc[cluster_0_data[strongest_feature].dropna().index, 'sum_vorp_4yrs'], 1)
        p = np.poly1d(z)
        x_trend = np.linspace(cluster_0_data[strongest_feature].min(),
                             cluster_0_data[strongest_feature].max(), 100)
        axes[1,1].plot(x_trend, p(x_trend), "r--", alpha=0.8, linewidth=2)
else:
    axes[1,1].text(0.5, 0.5, 'No correlations\navailable',
                  ha='center', va='center', transform=axes[1,1].transAxes)

plt.tight_layout()
plt.show()

# Additional analysis
print(f"\n=== CORRELATION ANALYSIS SUMMARY ===")
print(f"Total columns in dataset: {len(cluster_0_data.columns)}")
print(f"Numeric columns: {len(numeric_cols)}")
print(f"Non-numeric columns: {len(cluster_0_data.columns) - len(numeric_cols)}")

# Show some non-numeric columns as examples
non_numeric_cols = [col for col in cluster_0_data.columns if col not in numeric_cols]
if non_numeric_cols:
    print(f"Examples of non-numeric columns: {non_numeric_cols[:5]}")
    print("Sample values from non-numeric columns:")
    for col in non_numeric_cols[:3]:
        unique_vals = cluster_0_data[col].unique()[:5]
        print(f"  {col}: {list(unique_vals)}")

print(f"Features with correlation data: {len(correlations_clean)}")

if len(correlations_abs) > 0:
    print(f"Strongest positive correlation: {correlations_clean.max():.3f}")
    print(f"Strongest negative correlation: {correlations_clean.min():.3f}")
    print(f"Mean absolute correlation: {correlations_abs.mean():.3f}")

    # Count strong correlations
    strong_pos = (correlations_clean > 0.5).sum()
    strong_neg = (correlations_clean < -0.5).sum()
    strong_neg = (correlations_clean < -0.5).sum()
    moderate_pos = ((correlations_clean > 0.3) & (correlations_clean <= 0.5)).sum()
    moderate_neg = ((correlations_clean < -0.3) & (correlations_clean >= -0.5)).sum()

    print(f"\nCorrelation strength breakdown:")
    print(f"Strong positive (>0.5): {strong_pos}")
    print(f"Moderate positive (0.3-0.5): {moderate_pos}")
    print(f"Moderate negative (-0.5 to -0.3): {moderate_neg}")
    print(f"Strong negative (<-0.5): {strong_neg}")

    # Top features for potential modeling
    print(f"\n=== TOP FEATURES FOR MODELING ===")
    print("Features with |correlation| > 0.3:")
    strong_features = correlations_abs[correlations_abs > 0.3]
    for feature, corr_val in strong_features.items():
        actual_corr = correlations_clean[feature]
        print(f"  {feature}: {actual_corr:.3f}")

# Export correlation data
correlations_df = pd.DataFrame({
    'feature': correlations_clean.index,
    'correlation': correlations_clean.values,
    'abs_correlation': correlations_abs.values
}).sort_values('abs_correlation', ascending=False)

print(f"\n=== CORRELATION TABLE ===")
print(correlations_df.head(100))

print(f"\nCorrelation data saved to 'correlations_df' variable")
print(f"Access with: correlations_df")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Filter data for PlayStyleCluster = 0.0
cluster_2_data = final_df_transformed[final_df_transformed['PlayStyleCluster'] == 2.0].copy()

print(f"Filtered data shape: {cluster_2_data.shape}")

# Check data types
print(f"\nData types in dataset:")
print(cluster_2_data.dtypes.value_counts())

# Identify numeric columns only
numeric_cols = cluster_2_data.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns found: {len(numeric_cols)}")

# Check if target exists and is numeric
if 'sum_vorp_4yrs' not in numeric_cols:
    print("ERROR: sum_vorp_4yrs is not numeric or not found!")
    print("Available columns:", list(cluster_2_data.columns))
    print("sum_vorp_4yrs dtype:", cluster_2_data['sum_vorp_4yrs'].dtype if 'sum_vorp_4yrs' in cluster_2_data.columns else "Column not found")
else:
    print(f"Target variable (sum_vorp_4yrs) distribution:")
    print(cluster_2_data['sum_vorp_4yrs'].describe())

# Create numeric-only subset for correlations
numeric_data = cluster_2_data[numeric_cols]

# Calculate correlations with sum_vorp_4yrs (only if target is numeric)
if 'sum_vorp_4yrs' in numeric_cols:
    correlations = numeric_data.corr()['sum_vorp_4yrs'].sort_values(ascending=False)
else:
    print("Cannot calculate correlations - target variable not numeric")
    correlations = pd.Series(dtype=float)

if len(correlations) > 0:
    print("\n=== CORRELATIONS WITH sum_vorp_4yrs ===")
    print("Top positive correlations:")
    print(correlations.head(15))

    print("\nTop negative correlations:")
    print(correlations.tail(15))

    # Remove the target variable itself and any NaN correlations
    correlations_clean = correlations.drop('sum_vorp_4yrs').dropna()

    print(f"\n=== STRONGEST CORRELATIONS (excluding target) ===")
    print("Strongest absolute correlations:")
    correlations_abs = correlations_clean.abs().sort_values(ascending=False)
    print(correlations_abs.head(20))
else:
    print("\n=== NO CORRELATIONS AVAILABLE ===")
    print("Target variable may not be numeric or no numeric columns found")
    correlations_clean = pd.Series(dtype=float)
    correlations_abs = pd.Series(dtype=float)

# Create correlation matrix for visualization (numeric columns only)
corr_matrix = numeric_data.corr()

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

# 1. Target distribution
if 'sum_vorp_4yrs' in numeric_cols:
    axes[0,0].hist(cluster_2_data['sum_vorp_4yrs'], bins=20, alpha=0.7, edgecolor='black')
    axes[0,0].set_title('Distribution of sum_vorp_4yrs')
    axes[0,0].set_xlabel('sum_vorp_4yrs')
    axes[0,0].set_ylabel('Frequency')
else:
    axes[0,0].text(0.5, 0.5, 'sum_vorp_4yrs\nnot numeric',
                  ha='center', va='center', transform=axes[0,0].transAxes)
    axes[0,0].set_title('Target Variable Issue')

# 2. Correlation bar plot
top_corrs = correlations_clean.abs().sort_values(ascending=False).head(15)
top_corrs_signed = correlations_clean[top_corrs.index]

colors = ['red' if x < 0 else 'blue' for x in top_corrs_signed.values]
axes[0,1].barh(range(len(top_corrs_signed)), top_corrs_signed.values, color=colors, alpha=0.7)
axes[0,1].set_yticks(range(len(top_corrs_signed)))
axes[0,1].set_yticklabels(top_corrs_signed.index, fontsize=8)
axes[0,1].set_xlabel('Correlation with sum_vorp_4yrs')
axes[0,1].set_title('Top 15 Correlations with sum_vorp_4yrs')
axes[0,1].axvline(x=0, color='black', linestyle='-', alpha=0.3)

# 3. Correlation heatmap (top correlated features only)
if len(correlations_abs) > 0:
    # Get top 20 most correlated features + target
    top_features = ['sum_vorp_4yrs'] + list(correlations_abs.head(19).index)
    # Make sure all features exist in the data
    top_features = [f for f in top_features if f in cluster_2_data.columns]

    if len(top_features) > 1:
        subset_corr = cluster_2_data[top_features].corr()
        mask = np.triu(np.ones_like(subset_corr, dtype=bool))

        sns.heatmap(subset_corr, mask=mask, annot=True, cmap='RdBu_r', center=0,
                   square=True, ax=axes[1,0], fmt='.2f', cbar_kws={'shrink': 0.8})
        axes[1,0].set_title('Correlation Matrix (Top Features)')
    else:
        axes[1,0].text(0.5, 0.5, 'Not enough features\nfor correlation matrix',
                      ha='center', va='center', transform=axes[1,0].transAxes)

# 4. Scatter plot of strongest correlation
if len(correlations_abs) > 0:
    strongest_feature = correlations_abs.index[0]
    strongest_corr = correlations_clean[strongest_feature]

    axes[1,1].scatter(cluster_2_data[strongest_feature], cluster_2_data['sum_vorp_4yrs'],
                     alpha=0.6, s=50)
    axes[1,1].set_xlabel(strongest_feature)
    axes[1,1].set_ylabel('sum_vorp_4yrs')
    axes[1,1].set_title(f'Strongest Correlation: {strongest_feature}\nr = {strongest_corr:.3f}')

    # Add trend line
    if not cluster_2_data[strongest_feature].isna().all():
        z = np.polyfit(cluster_2_data[strongest_feature].dropna(),
                      cluster_2_data.loc[cluster_2_data[strongest_feature].dropna().index, 'sum_vorp_4yrs'], 1)
        p = np.poly1d(z)
        x_trend = np.linspace(cluster_2_data[strongest_feature].min(),
                             cluster_2_data[strongest_feature].max(), 100)
        axes[1,1].plot(x_trend, p(x_trend), "r--", alpha=0.8, linewidth=2)
else:
    axes[1,1].text(0.5, 0.5, 'No correlations\navailable',
                  ha='center', va='center', transform=axes[1,1].transAxes)

plt.tight_layout()
plt.show()

# Additional analysis
print(f"\n=== CORRELATION ANALYSIS SUMMARY ===")
print(f"Total columns in dataset: {len(cluster_2_data.columns)}")
print(f"Numeric columns: {len(numeric_cols)}")
print(f"Non-numeric columns: {len(cluster_2_data.columns) - len(numeric_cols)}")

# Show some non-numeric columns as examples
non_numeric_cols = [col for col in cluster_2_data.columns if col not in numeric_cols]
if non_numeric_cols:
    print(f"Examples of non-numeric columns: {non_numeric_cols[:5]}")
    print("Sample values from non-numeric columns:")
    for col in non_numeric_cols[:3]:
        unique_vals = cluster_2_data[col].unique()[:5]
        print(f"  {col}: {list(unique_vals)}")

print(f"Features with correlation data: {len(correlations_clean)}")

if len(correlations_abs) > 0:
    print(f"Strongest positive correlation: {correlations_clean.max():.3f}")
    print(f"Strongest negative correlation: {correlations_clean.min():.3f}")
    print(f"Mean absolute correlation: {correlations_abs.mean():.3f}")

    # Count strong correlations
    strong_pos = (correlations_clean > 0.5).sum()
    strong_neg = (correlations_clean < -0.5).sum()
    moderate_pos = ((correlations_clean > 0.3) & (correlations_clean <= 0.5)).sum()
    moderate_neg = ((correlations_clean < -0.3) & (correlations_clean >= -0.5)).sum()

    print(f"\nCorrelation strength breakdown:")
    print(f"Strong positive (>0.5): {strong_pos}")
    print(f"Moderate positive (0.3-0.5): {moderate_pos}")
    print(f"Moderate negative (-0.5 to -0.3): {moderate_neg}")
    print(f"Strong negative (<-0.5): {strong_neg}")

    # Top features for potential modeling
    print(f"\n=== TOP FEATURES FOR MODELING ===")
    print("Features with |correlation| > 0.3:")
    strong_features = correlations_abs[correlations_abs > 0.3]
    for feature, corr_val in strong_features.items():
        actual_corr = correlations_clean[feature]
        print(f"  {feature}: {actual_corr:.3f}")

# Export correlation data
correlations_df = pd.DataFrame({
    'feature': correlations_clean.index,
    'correlation': correlations_clean.values,
    'abs_correlation': correlations_abs.values
}).sort_values('abs_correlation', ascending=False)

print(f"\n=== CORRELATION TABLE ===")
print(correlations_df.head(100))

print(f"\nCorrelation data saved to 'correlations_df' variable")
print(f"Access with: correlations_df")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Filter data for PlayStyleCluster = 0.0
cluster_1_data = final_df_transformed[final_df_transformed['PlayStyleCluster'] == 1.0].copy()

print(f"Filtered data shape: {cluster_1_data.shape}")

# Check data types
print(f"\nData types in dataset:")
print(cluster_1_data.dtypes.value_counts())

# Identify numeric columns only
numeric_cols = cluster_1_data.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumeric columns found: {len(numeric_cols)}")

# Check if target exists and is numeric
if 'sum_vorp_4yrs' not in numeric_cols:
    print("ERROR: sum_vorp_4yrs is not numeric or not found!")
    print("Available columns:", list(cluster_1_data.columns))
    print("sum_vorp_4yrs dtype:", cluster_1_data['sum_vorp_4yrs'].dtype if 'sum_vorp_4yrs' in cluster_1_data.columns else "Column not found")
else:
    print(f"Target variable (sum_vorp_4yrs) distribution:")
    print(cluster_1_data['sum_vorp_4yrs'].describe())

# Create numeric-only subset for correlations
numeric_data = cluster_1_data[numeric_cols]

# Calculate correlations with sum_vorp_4yrs (only if target is numeric)
if 'sum_vorp_4yrs' in numeric_cols:
    correlations = numeric_data.corr()['sum_vorp_4yrs'].sort_values(ascending=False)
else:
    print("Cannot calculate correlations - target variable not numeric")
    correlations = pd.Series(dtype=float)

if len(correlations) > 0:
    print("\n=== CORRELATIONS WITH sum_vorp_4yrs ===")
    print("Top positive correlations:")
    print(correlations.head(15))

    print("\nTop negative correlations:")
    print(correlations.tail(15))

    # Remove the target variable itself and any NaN correlations
    correlations_clean = correlations.drop('sum_vorp_4yrs').dropna()

    print(f"\n=== STRONGEST CORRELATIONS (excluding target) ===")
    print("Strongest absolute correlations:")
    correlations_abs = correlations_clean.abs().sort_values(ascending=False)
    print(correlations_abs.head(20))
else:
    print("\n=== NO CORRELATIONS AVAILABLE ===")
    print("Target variable may not be numeric or no numeric columns found")
    correlations_clean = pd.Series(dtype=float)
    correlations_abs = pd.Series(dtype=float)

# Create correlation matrix for visualization (numeric columns only)
corr_matrix = numeric_data.corr()

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

# 1. Target distribution
if 'sum_vorp_4yrs' in numeric_cols:
    axes[0,0].hist(cluster_1_data['sum_vorp_4yrs'], bins=20, alpha=0.7, edgecolor='black')
    axes[0,0].set_title('Distribution of sum_vorp_4yrs')
    axes[0,0].set_xlabel('sum_vorp_4yrs')
    axes[0,0].set_ylabel('Frequency')
else:
    axes[0,0].text(0.5, 0.5, 'sum_vorp_4yrs\nnot numeric',
                  ha='center', va='center', transform=axes[0,0].transAxes)
    axes[0,0].set_title('Target Variable Issue')

# 2. Correlation bar plot
top_corrs = correlations_clean.abs().sort_values(ascending=False).head(15)
top_corrs_signed = correlations_clean[top_corrs.index]

colors = ['red' if x < 0 else 'blue' for x in top_corrs_signed.values]
axes[0,1].barh(range(len(top_corrs_signed)), top_corrs_signed.values, color=colors, alpha=0.7)
axes[0,1].set_yticks(range(len(top_corrs_signed)))
axes[0,1].set_yticklabels(top_corrs_signed.index, fontsize=8)
axes[0,1].set_xlabel('Correlation with sum_vorp_4yrs')
axes[0,1].set_title('Top 15 Correlations with sum_vorp_4yrs')
axes[0,1].axvline(x=0, color='black', linestyle='-', alpha=0.3)

# 3. Correlation heatmap (top correlated features only)
if len(correlations_abs) > 0:
    # Get top 20 most correlated features + target
    top_features = ['sum_vorp_4yrs'] + list(correlations_abs.head(19).index)
    # Make sure all features exist in the data
    top_features = [f for f in top_features if f in cluster_1_data.columns]

    if len(top_features) > 1:
        subset_corr = cluster_1_data[top_features].corr()
        mask = np.triu(np.ones_like(subset_corr, dtype=bool))

        sns.heatmap(subset_corr, mask=mask, annot=True, cmap='RdBu_r', center=0,
                   square=True, ax=axes[1,0], fmt='.2f', cbar_kws={'shrink': 0.8})
        axes[1,0].set_title('Correlation Matrix (Top Features)')
    else:
        axes[1,0].text(0.5, 0.5, 'Not enough features\nfor correlation matrix',
                      ha='center', va='center', transform=axes[1,0].transAxes)

# 4. Scatter plot of strongest correlation
if len(correlations_abs) > 0:
    strongest_feature = correlations_abs.index[0]
    strongest_corr = correlations_clean[strongest_feature]

    axes[1,1].scatter(cluster_1_data[strongest_feature], cluster_1_data['sum_vorp_4yrs'],
                     alpha=0.6, s=50)
    axes[1,1].set_xlabel(strongest_feature)
    axes[1,1].set_ylabel('sum_vorp_4yrs')
    axes[1,1].set_title(f'Strongest Correlation: {strongest_feature}\nr = {strongest_corr:.3f}')

    # Add trend line
    if not cluster_1_data[strongest_feature].isna().all():
        z = np.polyfit(cluster_1_data[strongest_feature].dropna(),
                      cluster_1_data.loc[cluster_1_data[strongest_feature].dropna().index, 'sum_vorp_4yrs'], 1)
        p = np.poly1d(z)
        x_trend = np.linspace(cluster_1_data[strongest_feature].min(),
                             cluster_1_data[strongest_feature].max(), 100)
        axes[1,1].plot(x_trend, p(x_trend), "r--", alpha=0.8, linewidth=2)
else:
    axes[1,1].text(0.5, 0.5, 'No correlations\navailable',
                  ha='center', va='center', transform=axes[1,1].transAxes)

plt.tight_layout()
plt.show()

# Additional analysis
print(f"\n=== CORRELATION ANALYSIS SUMMARY ===")
print(f"Total columns in dataset: {len(cluster_1_data.columns)}")
print(f"Numeric columns: {len(numeric_cols)}")
print(f"Non-numeric columns: {len(cluster_1_data.columns) - len(numeric_cols)}")

# Show some non-numeric columns as examples
non_numeric_cols = [col for col in cluster_1_data.columns if col not in numeric_cols]
if non_numeric_cols:
    print(f"Examples of non-numeric columns: {non_numeric_cols[:5]}")
    print("Sample values from non-numeric columns:")
    for col in non_numeric_cols[:3]:
        unique_vals = cluster_1_data[col].unique()[:5]
        print(f"  {col}: {list(unique_vals)}")

print(f"Features with correlation data: {len(correlations_clean)}")

if len(correlations_abs) > 0:
    print(f"Strongest positive correlation: {correlations_clean.max():.3f}")
    print(f"Strongest negative correlation: {correlations_clean.min():.3f}")
    print(f"Mean absolute correlation: {correlations_abs.mean():.3f}")

    # Count strong correlations
    strong_pos = (correlations_clean > 0.5).sum()
    strong_neg = (correlations_clean < -0.5).sum()
    moderate_pos = ((correlations_clean > 0.3) & (correlations_clean <= 0.5)).sum()
    moderate_neg = ((correlations_clean < -0.3) & (correlations_clean >= -0.5)).sum()

    print(f"\nCorrelation strength breakdown:")
    print(f"Strong positive (>0.5): {strong_pos}")
    print(f"Moderate positive (0.3-0.5): {moderate_pos}")
    print(f"Moderate negative (-0.5 to -0.3): {moderate_neg}")
    print(f"Strong negative (<-0.5): {strong_neg}")

    # Top features for potential modeling
    print(f"\n=== TOP FEATURES FOR MODELING ===")
    print("Features with |correlation| > 0.3:")
    strong_features = correlations_abs[correlations_abs > 0.3]
    for feature, corr_val in strong_features.items():
        actual_corr = correlations_clean[feature]
        print(f"  {feature}: {actual_corr:.3f}")

# Export correlation data
correlations_df = pd.DataFrame({
    'feature': correlations_clean.index,
    'correlation': correlations_clean.values,
    'abs_correlation': correlations_abs.values
}).sort_values('abs_correlation', ascending=False)

print(f"\n=== CORRELATION TABLE ===")
print(correlations_df.head(100))

print(f"\nCorrelation data saved to 'correlations_df' variable")
print(f"Access with: correlations_df")

In [ ]:
column_averages = cluster_0_data.mean(numeric_only=True).reset_index()
column_averages.columns = ["Column", "Average Value"]
column_averages


In [ ]:
final_df_transform["Actual"] = (final_df_transform["sum_vorp_4yrs"] > 4).astype(int)


In [ ]:
column_averages = cluster_1_data.mean(numeric_only=True).reset_index()
column_averages.columns = ["Column", "Average Value"]
column_averages

In [ ]:
column_averages = cluster_2_data.mean(numeric_only=True).reset_index()
column_averages.columns = ["Column", "Average Value"]
column_averages

In [ ]:
# Add 'Actual' column early, BEFORE modeling
cluster_0_data["Actual"] = (cluster_0_data["sum_vorp_4yrs"] > 4).astype(int)


In [ ]:
cluster_0_data.head(5)
cluster_0_data['close_min'] = cluster_0_data['close_makes'] / cluster_0_data['Min%']
cluster_1_data['close_min'] = cluster_1_data['close_makes'] / cluster_1_data['Min%']
cluster_2_data['close_min'] = cluster_2_data['close_makes'] / cluster_2_data['Min%']

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

# === Step 0: Define features and target ===
selected_features = ['DraftValue', 'LogDR',  'DBPM', 'LogUsg']
X_all = cluster_0_data[selected_features].copy().fillna(cluster_0_data[selected_features].mean())
y_all = cluster_0_data["Actual"]

# === Step 1: Multi-seed logistic regression ===
all_metrics = []
all_coefs = []
saved_scaler = None

for seed in range(20):
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.2, random_state=seed, stratify=y_all
    )

    if len(np.unique(y_train)) < 2:
        print(f"Skipping seed {seed}: only one class in y_train.")
        continue

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    if saved_scaler is None:
        saved_scaler = scaler

    model = LogisticRegression(class_weight={0: 1, 1: 3}, max_iter=1000)
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    all_metrics.append({
        'seed': seed,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_prob)
    })

    all_coefs.append(model.coef_[0])

# === Step 2: Average metrics and coefficients ===
metrics_df = pd.DataFrame(all_metrics)
avg_metrics = metrics_df.mean(numeric_only=True)
avg_coefs = np.mean(all_coefs, axis=0)

print("\n=== AVERAGED METRICS OVER 10 SEEDS ===")
for k, v in avg_metrics.items():
    print(f"{k}: {v:.4f}")

# === Step 3: Predict probabilities on full dataset ===
X_scaled_full = saved_scaler.transform(X_all)
logits = np.dot(X_scaled_full, avg_coefs)
probabilities = 1 / (1 + np.exp(-logits))
cluster_0_data["Prediction"] = probabilities

# === Step 4: Show top 10 players ===
top_10 = cluster_0_data[["Name", "Prediction", "Actual"]].sort_values("Prediction", ascending=False).head(50)

print("\nTop 10 Predicted Players:")
print(top_10.to_string(index=False))


In [ ]:
# Combine feature names with average coefficients
coef_df = pd.DataFrame({
    "Feature": selected_features,
    "Avg_Coefficient": avg_coefs,
    "Abs_Coefficient": np.abs(avg_coefs)
}).sort_values("Abs_Coefficient", ascending=False)
print("\n=== AVERAGE COEFFICIENTS ===")
print(coef_df.to_string(index=False))

In [ ]:

models_by_cluster = {}

models_by_cluster[0.0] = {
    "features": selected_features,
    "scaler": saved_scaler,
    "avg_coefs": avg_coefs,
    "df_with_predictions": cluster_0_data[["Name", "Prediction", "Actual",  "Year", "PlayStyleCluster"]].copy()
}


In [ ]:
# Add 'Actual' column early, BEFORE modeling
cluster_1_data["Actual"] = (cluster_1_data["sum_vorp_4yrs"] > 4).astype(int)

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
  accuracy_score, precision_score, recall_score,
  f1_score, roc_auc_score
)

# === Step 0: Define features and target ===
selected_features = ['LogUsg/Ast', 'DraftValue', 'LogREB', 'LogFT%','TO']
X_all = cluster_1_data[selected_features].copy().fillna(cluster_1_data[selected_features].mean())
y_all = cluster_1_data["Actual"]

# === Step 1: Multi-seed logistic regression ===
all_metrics = []
all_coefs = []
saved_scaler = None

for seed in range(20):
  X_train, X_test, y_train, y_test = train_test_split(
      X_all, y_all, test_size=0.2, random_state=seed, stratify=y_all
  )

  if len(np.unique(y_train)) < 2:
      print(f"Skipping seed {seed}: only one class in y_train.")
      continue

  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_test_scaled = scaler.transform(X_test)

  if saved_scaler is None:
      saved_scaler = scaler

  model = LogisticRegression(class_weight={0: 1, 1: 3}, max_iter=1000)
  model.fit(X_train_scaled, y_train)

  y_pred = model.predict(X_test_scaled)
  y_prob = model.predict_proba(X_test_scaled)[:, 1]

  all_metrics.append({
      'seed': seed,
      'accuracy': accuracy_score(y_test, y_pred),
      'precision': precision_score(y_test, y_pred, zero_division=0),
      'recall': recall_score(y_test, y_pred, zero_division=0),
      'f1': f1_score(y_test, y_pred, zero_division=0),
      'roc_auc': roc_auc_score(y_test, y_prob)
  })

  all_coefs.append(model.coef_[0])

# === Step 2: Average metrics and coefficients ===
metrics_df = pd.DataFrame(all_metrics)
avg_metrics = metrics_df.mean(numeric_only=True)
avg_coefs = np.mean(all_coefs, axis=0)

print("\n=== AVERAGED METRICS OVER 10 SEEDS ===")
for k, v in avg_metrics.items():
  print(f"{k}: {v:.4f}")

# === Step 3: Predict probabilities on full dataset ===
X_scaled_full = saved_scaler.transform(X_all)
logits = np.dot(X_scaled_full, avg_coefs)
probabilities = 1 / (1 + np.exp(-logits))
cluster_1_data["Prediction"] = probabilities

# Adjust predictions for wings (cluster 1.0) based on class year
if 'Player_Encoded' in cluster_1_data.columns:
  sr_mask = cluster_1_data['Player_Encoded'] == 4
  jr_mask = cluster_1_data['Player_Encoded'] == 3
  so_mask = cluster_1_data['Player_Encoded'] == 2

  cluster_1_data.loc[sr_mask, 'Prediction'] = np.maximum(0, cluster_1_data.loc[sr_mask, 'Prediction'] - 0.1)
  cluster_1_data.loc[jr_mask, 'Prediction'] = np.maximum(0, cluster_1_data.loc[jr_mask, 'Prediction'] - 0.05)
  cluster_1_data.loc[so_mask, 'Prediction'] = np.maximum(0, cluster_1_data.loc[so_mask, 'Prediction'] - 0.025)

  print("\nApplied class year adjustment for wings:")
  print(f"  Seniors: -0.1")
  print(f"  Juniors: -0.05")
  print(f"  Sophomores: -0.025")

# === Step 4: Show top 10 players by prediction ===
top_10 = (cluster_1_data[["Name", "Prediction", "Actual"]].sort_values(
  "Prediction", ascending=False
).head(50))

print("\nTop 50 Predicted Players:")
print(top_10.to_string(index=False))


In [ ]:
# Combine feature names with average coefficients
coef_df = pd.DataFrame({
    "Feature": selected_features,
    "Avg_Coefficient": avg_coefs,
    "Abs_Coefficient": np.abs(avg_coefs)
}).sort_values("Abs_Coefficient", ascending=False)
print("\n=== AVERAGE COEFFICIENTS ===")
print(coef_df.to_string(index=False))


In [ ]:


models_by_cluster[1.0] = {
    "features": selected_features,
    "scaler": saved_scaler,
    "avg_coefs": avg_coefs,
    "df_with_predictions": cluster_1_data[["Name", "Prediction", "Actual",  "Year", "PlayStyleCluster"]].copy()
}


In [ ]:
# Add 'Actual' column early, BEFORE modeling
cluster_2_data["Actual"] = (cluster_2_data["sum_vorp_4yrs"] > 4 ).astype(int)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

cluster_2_data['close_min'] = cluster_2_data['close_makes'] / cluster_2_data['Min%']

# === Step 0: Define features and target ===
selected_features = ['LogFT%',  'DraftValue', 'close_min', 'Close 2 %']
X_all = cluster_2_data[selected_features].copy().fillna(cluster_2_data[selected_features].mean())
y_all = cluster_2_data["Actual"]

# === Step 1: Multi-seed logistic regression ===
all_metrics = []
all_coefs = []
saved_scaler = None

for seed in range(20):
    X_train, X_test, y_train, y_test = train_test_split(
        X_all, y_all, test_size=0.2, random_state=seed, stratify=y_all
    )

    if len(np.unique(y_train)) < 2:
        print(f"Skipping seed {seed}: only one class in y_train.")
        continue

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    if saved_scaler is None:
        saved_scaler = scaler

    model = LogisticRegression(class_weight={0: 1, 1: 3}, max_iter=1000)
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    all_metrics.append({
        'seed': seed,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_prob)
    })

    all_coefs.append(model.coef_[0])

# === Step 2: Average metrics and coefficients ===
metrics_df = pd.DataFrame(all_metrics)
avg_metrics = metrics_df.mean(numeric_only=True)
avg_coefs = np.mean(all_coefs, axis=0)

print("\n=== AVERAGED METRICS OVER 10 SEEDS ===")
for k, v in avg_metrics.items():
    print(f"{k}: {v:.4f}")

# === Step 3: Predict probabilities on full dataset ===
X_scaled_full = saved_scaler.transform(X_all)
logits = np.dot(X_scaled_full, avg_coefs)
probabilities = 1 / (1 + np.exp(-logits))
cluster_2_data["Prediction"] = probabilities

# === Step 4: Show top 10 players by prediction ===
top_10 = (cluster_2_data[["Name", "Prediction", "Actual"]].sort_values(
  "Prediction", ascending=False
).head(50))

print("\nTop 50 Predicted Players:")
print(top_10.to_string(index=False))




In [ ]:
# Combine feature names with average coefficients
coef_df = pd.DataFrame({
    "Feature": selected_features,
    "Avg_Coefficient": avg_coefs,
    "Abs_Coefficient": np.abs(avg_coefs)
}).sort_values("Abs_Coefficient", ascending=False)
print("\n=== AVERAGE COEFFICIENTS ===")
print(coef_df.to_string(index=False))

In [ ]:


models_by_cluster[2.0] = {
    "features": selected_features,
    "scaler": saved_scaler,
    "avg_coefs": avg_coefs,
    "df_with_predictions": cluster_2_data[["Name", "Prediction", "Actual",  "Year", "PlayStyleCluster"]].copy()
}

In [ ]:
def show_clustered_player_prediction(player_name):
    row = final_df_transform[final_df_transform["Name"] == player_name]

    if row.empty:
        print(f"Player '{player_name}' not found.")
        return

    cluster = row["PlayStyleCluster"].iloc[0]

    if cluster not in models_by_cluster:
        print(f"No model found for cluster {cluster}.")
        return

    model_data = models_by_cluster[cluster]
    df = model_data["df_with_predictions"]

    match = df[df["Name"] == player_name]
    if not match.empty:
        prob = match["Prediction"].iloc[0]
        actual = match["Actual"].iloc[0]
        print(f"\nPrediction for {player_name} (Cluster {cluster}):")
        print(f"  Probability VORP > 6: {prob:.4f}")
        print(f"  Actual Outcome     : {actual}")
        return

    # Fallback: compute prediction on-the-fly
    features = model_data["features"]
    scaler = model_data["scaler"]
    avg_coefs = model_data["avg_coefs"]

    player_features = row[features].fillna(0)
    player_scaled = scaler.transform(row[features])



    logit = np.dot(player_scaled, avg_coefs)
    prob = 1 / (1 + np.exp(-logit))
    actual = row["Actual"].iloc[0]

    print(f"\nPrediction for {player_name} (Cluster {cluster}):")
    print(f"  Probability VORP > 6: {prob[0]:.4f} (computed on-the-fly)")
    print(f"  Actual Outcome     : {actual}")


In [ ]:
def adjust_and_show_prediction(player_name):
    # Simply call the original function - no age adjustment needed since Player_Encoded is a model feature
    show_clustered_player_prediction(player_name)

In [ ]:
print("Models available:", list(models_by_cluster.keys()))


In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)  # Show ALL columns

final_df_transform.head(20)

In [ ]:
for cluster, model_data in models_by_cluster.items():
    df = model_data["df_with_predictions"]
    print(f"Cluster {cluster}: {len(df)} players")
    print(df[["Year", "Prediction"]].head(10))  # sanity check


In [ ]:
# Recompute df_with_predictions using ALL players from final_df_transform
for cluster in [0.0, 1.0, 2.0]:
  model_data = models_by_cluster[cluster]
  features = model_data["features"]
  scaler = model_data["scaler"]
  avg_coefs = model_data["avg_coefs"]

  # Use ALL rows from final_df_transform with matching cluster
  df_cluster = final_df_transform[final_df_transform["PlayStyleCluster"] == cluster].copy()

  df_cluster[features] = df_cluster[features].fillna(0)
  X_scaled = scaler.transform(df_cluster[features])
  logits = np.dot(X_scaled, avg_coefs)
  probabilities = 1 / (1 + np.exp(-logits))

  df_cluster["Prediction"] = probabilities

  # Adjust predictions for wings (cluster 1.0) based on class year
  if cluster == 1.0 and 'Player_Encoded' in df_cluster.columns:
      sr_mask = df_cluster['Player_Encoded'] == 4
      jr_mask = df_cluster['Player_Encoded'] == 3
      so_mask = df_cluster['Player_Encoded'] == 2

      df_cluster.loc[sr_mask, 'Prediction'] = np.maximum(0, df_cluster.loc[sr_mask, 'Prediction'] - 0.1)
      df_cluster.loc[jr_mask, 'Prediction'] = np.maximum(0, df_cluster.loc[jr_mask, 'Prediction'] - 0.05)
      df_cluster.loc[so_mask, 'Prediction'] = np.maximum(0, df_cluster.loc[so_mask, 'Prediction'] - 0.025)

  if cluster == 0.0 and 'Player_Encoded' in df_cluster.columns:
      sr_mask = df_cluster['Player_Encoded'] == 4
      jr_mask = df_cluster['Player_Encoded'] == 3
      so_mask = df_cluster['Player_Encoded'] == 2

      df_cluster.loc[sr_mask, 'Prediction'] = np.maximum(0, df_cluster.loc[sr_mask, 'Prediction'] - 0.1)
      df_cluster.loc[jr_mask, 'Prediction'] = np.maximum(0, df_cluster.loc[jr_mask, 'Prediction'] - 0.05)
      df_cluster.loc[so_mask, 'Prediction'] = np.maximum(0, df_cluster.loc[so_mask, 'Prediction'] - 0.025)

  if cluster == 2.0 and 'Player_Encoded' in df_cluster.columns:
      sr_mask = df_cluster['Player_Encoded'] == 4
      jr_mask = df_cluster['Player_Encoded'] == 3
      so_mask = df_cluster['Player_Encoded'] == 2

      df_cluster.loc[sr_mask, 'Prediction'] = np.maximum(0, df_cluster.loc[sr_mask, 'Prediction'] - 0.1)
      df_cluster.loc[jr_mask, 'Prediction'] = np.maximum(0, df_cluster.loc[jr_mask, 'Prediction'] - 0.05)
      df_cluster.loc[so_mask, 'Prediction'] = np.maximum(0, df_cluster.loc[so_mask, 'Prediction'] - 0.025)
  # Fill "Actual" with None if it doesn't exist
  if "Actual" not in df_cluster.columns:
      df_cluster["Actual"] = None

  # Save predictions
  models_by_cluster[cluster]["df_with_predictions"] = df_cluster[
      ["Name", "Year", "Prediction", "Actual", "PlayStyleCluster"]
  ].copy()


In [ ]:
def show_top_predictions_by_year(year, top_n=100):
    combined_rows = []

    for cluster, model_data in models_by_cluster.items():
        df = model_data.get("df_with_predictions")
        if df is None or "Year" not in df.columns:
            continue

        df_year = df[df["Year"] == year].copy()
        if df_year.empty:
            continue

        # No age adjustment needed since Player_Encoded is a feature in the model now
        df_year["Cluster"] = cluster
        combined_rows.append(df_year[["Name", "Year", "Prediction", "Actual", "Cluster"]])

    if not combined_rows:
        print(f"No data found for year {year}.")
        return

    all_preds = pd.concat(combined_rows)
    top_preds = all_preds.sort_values("Prediction", ascending=False).head(top_n)

    print(f"\nTop {top_n} Predicted Players from Year {year}:")
    print(top_preds.to_string(index=False))

In [ ]:
final_df_transform.columns

In [ ]:
show_top_predictions_by_year(26)


In [ ]:
print(models_by_cluster[1.0]["features"])


In [ ]:
print(cluster_1_data.head()[["DR", "LogDR"]])


In [ ]:
def explain_manual_prediction(cluster, raw_inputs: dict):
      # === Load model data ===
      model_data = models_by_cluster[cluster]
      features = model_data["features"]
      scaler = model_data["scaler"]
      avg_coefs = model_data["avg_coefs"]

      # DEBUG: Print scaler mean and std being used
      print(f"\n[DEBUG] Cluster: {cluster}")
      print(f"[DEBUG] Scaler Mean: {scaler.mean_}")
      print(f"[DEBUG] Scaler Scale: {scaler.scale_}")

      # === Build input vector ===
      input_vector = {feat: raw_inputs.get(feat, 0.0) for feat in features}
      df_input = pd.DataFrame([input_vector])

      # === Scale input and compute dot product ===
      X_scaled = scaler.transform(df_input)[0]
      contributions = X_scaled * avg_coefs
      logit = np.sum(contributions)
      prob = 1 / (1 + np.exp(-logit))

      # === Print feature-by-feature breakdown ===
      print(f"\nFeature Contributions for Cluster {cluster}:")
      for feat, raw_val, scaled_val, coef, contrib in zip(features, df_input.iloc[0], X_scaled, avg_coefs, contributions):
          print(f"{feat:<25} Raw: {raw_val:>7.4f} | Scaled: {scaled_val:>7.4f} | Coef: {coef:>7.4f} | Contribution: {contrib:>7.4f}")

      print(f"\nLogit Total: {logit:.4f}")
      print(f"Prediction (before adjustment): {prob:.4f}")

      # Adjust for wings (cluster 1.0) based on Player_Encoded
      if cluster == 1.0 and 'Player_Encoded' in raw_inputs:
          player_enc = raw_inputs['Player_Encoded']
          adjustment = 0
          if player_enc == 4:  # Senior
              adjustment = -0.1
          elif player_enc == 3:  # Junior
              adjustment = -0.05
          elif player_enc == 2:  # Sophomore
              adjustment = -0.025

          if adjustment != 0:
              prob = max(0, prob + adjustment)
              print(f"Class year adjustment: {adjustment}")

      if cluster == 0.0 and 'Player_Encoded' in raw_inputs:
          player_enc = raw_inputs['Player_Encoded']
          adjustment = 0
          if player_enc == 4:  # Senior
              adjustment = -0.1
          elif player_enc == 3:  # Junior
              adjustment = -0.05
          elif player_enc == 2:  # Sophomore
              adjustment = -0.025

          if adjustment != 0:
              prob = max(0, prob + adjustment)
              print(f"Class year adjustment: {adjustment}")

      if cluster == 2.0 and 'Player_Encoded' in raw_inputs:
          player_enc = raw_inputs['Player_Encoded']
          adjustment = 0
          if player_enc == 4:  # Senior
              adjustment = -0.1
          elif player_enc == 3:  # Junior
              adjustment = -0.05
          elif player_enc == 2:  # Sophomore
              adjustment = -0.025

          if adjustment != 0:
              prob = max(0, prob + adjustment)
              print(f"Class year adjustment: {adjustment}")
      print(f"Final Prediction: {prob:.4f}")


In [ ]:
manual_inputs = {
    'LogUsg/Ast': 1.0296194171812,
    'DraftValue':0.555763,
    'LogREB': 3.3068867021909,
    'LogFT%': 0.55331023537215,
    'TO': 13.1,
    'Player_Encoded': 1
}
explain_manual_prediction(1.0, manual_inputs)



In [ ]:
manual_inputs = {
    'DraftValue':0.98,
    'LogDR':2.9549102790337,
    'DBPM':3,
    'LogUsg':3.1354942159291,
    'Player_Encoded': 1
}

explain_manual_prediction(0.0, manual_inputs)


In [ ]:
manual_inputs = {
    "LogFT%":  0.6119371251344,
    "DraftValue": 0.703952,
    "close_min": 1.070359,
    "Close 2 %": .62,
    'Player_Encoded': 1
}
explain_manual_prediction(2.0, manual_inputs)

In [ ]:
# View all features for Shai Gilgeous-Alexander (case-insensitive)
shai_row = final_df_transform[final_df_transform["Name"].str.contains("shai", case=False, na=False)]

# Transpose for readability (features as rows)
pd.set_option("display.max_rows", 200)  # Show more rows if needed
print(shai_row.T)


In [ ]:
# Let's say the target column is "sum_4yr_vorp"
average_by_cluster = final_df_filtered.groupby("PlayStyleCluster")[("zscore_vorp")].mean()
std_by_cluster = final_df_filtered.groupby("PlayStyleCluster")[("zscore_vorp")].std()
print(average_by_cluster)
print(std_by_cluster)



In [ ]:
# Step 1: Create a binary column for VORP > 4
final_df_filtered["vorp_gt_4"] = final_df_filtered["sum_vorp_4yrs"] > 4

# Step 2: Group by cluster and calculate the mean of the boolean (which gives the %)
percentages = final_df_filtered.groupby("PlayStyleCluster")["vorp_gt_4"].mean() * 100

# Step 3: Display as rounded %
print(percentages.round(2))


In [ ]:
import pickle

# Export your final dataframe
final_df_transform.to_csv('final_df_transform.csv', index=False)
print("Exported final_df_transform.csv")

# Export your models dictionary
with open('models_by_cluster.pkl', 'wb') as f:
    pickle.dump(models_by_cluster, f)
print("Exported models_by_cluster.pkl")


In [ ]:
import sys
!{sys.executable} -m pip install flask


In [ ]:
 # Export predictions to CSV for streamlit app (2019-2025 only)
import pandas as pd
import numpy as np

def apply_class_adjustment(cluster, prob, player_encoded):
  """Apply class year adjustment for all clusters"""
  if cluster in [0.0, 1.0, 2.0]:
      if player_encoded == 4:  # Senior
          return max(0, prob - 0.1)
      elif player_encoded == 3:  # Junior
          return max(0, prob - 0.05)
      elif player_encoded == 2:  # Sophomore
          return max(0, prob - 0.025)
  return prob

# Calculate predictions for all players from 2019 onwards
all_predictions = []

# Filter for years 2019 and later (Year >= 19)
recent_players = final_df_transform[final_df_transform['Year'] >= 26]

for idx, player in recent_players.iterrows():
  cluster = player["PlayStyleCluster"]

  if cluster in models_by_cluster:
      model_data = models_by_cluster[cluster]
      features = model_data["features"]
      scaler = model_data["scaler"]
      avg_coefs = model_data["avg_coefs"]

      # Calculate prediction
      player_features = player[features].fillna(0)
      player_scaled = scaler.transform(player_features.values.reshape(1, -1))
      logit = np.dot(player_scaled, avg_coefs)
      prob = 1 / (1 + np.exp(-logit))

      # Apply class year adjustment
      adjusted_prob = apply_class_adjustment(cluster, prob[0], player.get('Player_Encoded', 0))

      all_predictions.append({
          'Name': player['Name'],
          'Year': 2000 + player['Year'],
          'Prediction': adjusted_prob,
      })

# Create DataFrame
predictions_df = pd.DataFrame(all_predictions)

# Sort by prediction score (highest to lowest)
predictions_df = predictions_df.sort_values('Prediction', ascending=False)

# Export to CSV
predictions_df.to_csv('ncaab_predictions2026.csv', index=False)
print(f"Exported {len(predictions_df)} predictions to ncaab_predictions2026.csv (2026)")

# Show top 10
print("\nTop 10 predictions:")
print(predictions_df.head(30)[['Name', 'Year', 'Prediction']])



In [ ]:
from scipy.spatial.distance import cdist

def find_similar_players(player_name, top_n=10, include_self=False):
    """
    Find the most similar historical prospects to a given player,
    based on Euclidean distance using model features + Height_in, Pick,
    Player_Encoded, and model prediction score.
    """
    row = final_df_transform[final_df_transform['Name'].str.contains(player_name, case=False, na=False)]

    if row.empty:
        print(f"Player '{player_name}' not found.")
        return

    if len(row) > 1:
        print(f"Multiple matches:")
        print(row[['Name', 'Year', 'PlayStyleCluster']].to_string())
        print(f"\nUsing first match: {row['Name'].iloc[0]}")
        row = row.iloc[[0]]

    cluster = row['PlayStyleCluster'].iloc[0]
    model_data = models_by_cluster[cluster]
    features = model_data['features']

    extra_features = [f for f in ['Height_in', 'Player_Encoded'] if f in final_df_transform.columns]
    sim_features = features + [f for f in extra_features if f not in features]

    cluster_df = final_df_transform[final_df_transform['PlayStyleCluster'] == cluster].copy()
    cluster_df[sim_features] = cluster_df[sim_features].fillna(0)

    # Merge in model prediction score
    pred_df = model_data['df_with_predictions'][['Name', 'Prediction']].copy()
    cluster_df = cluster_df.merge(pred_df.rename(columns={'Prediction': '_model_score'}), on='Name', how='left')
    cluster_df['_model_score'] = cluster_df['_model_score'].fillna(0)
    sim_features_with_score = sim_features + ['_model_score']

    # Get target player's model score
    target_score = pred_df.loc[pred_df['Name'] == row['Name'].iloc[0], 'Prediction']
    row = row.copy()
    row['_model_score'] = target_score.iloc[0] if not target_score.empty else 0.0

    sim_scaler = StandardScaler()
    X_scaled = sim_scaler.fit_transform(cluster_df[sim_features_with_score])
    target_scaled = sim_scaler.transform(row[sim_features_with_score].fillna(0))

    distances = cdist(target_scaled, X_scaled, metric='euclidean')[0]
    cluster_df['_distance'] = distances
    sorted_df = cluster_df.sort_values('_distance')

    if not include_self:
        sorted_df = sorted_df[~sorted_df['Name'].str.lower().eq(row['Name'].iloc[0].lower())]

    pred_lookup = pred_df.set_index('Name')

    cluster_labels = {0.0: 'Centers', 1.0: 'Wings/Forwards', 2.0: 'Guards'}
    print(f"\nMost similar prospects to {row['Name'].iloc[0]} "
          f"({row['Year'].iloc[0] if 'Year' in row.columns else '?'}, "
          f"{cluster_labels.get(cluster, f'Cluster {cluster}')})")
    print(f"Features used: {sim_features_with_score}\n")

    results = []
    for _, sim_row in sorted_df.head(top_n).iterrows():
        name = sim_row['Name']
        year = sim_row.get('Year', '?')
        dist = sim_row['_distance']
        vorp = sim_row.get('sum_vorp_4yrs', None)
        pick = sim_row.get('Pick', None)
        height = sim_row.get('Height_in', None)
        enc = sim_row.get('Player_Encoded', None)
        pred = pred_lookup.loc[name, 'Prediction'] if name in pred_lookup.index else None
        actual = model_data['df_with_predictions'].set_index('Name').loc[name, 'Actual'] if name in model_data['df_with_predictions']['Name'].values else None
        results.append({
            'Name': name,
            'Year': year,
            'Pick': pick,
            'Height': round(height, 1) if pd.notna(height) else None,
            'Class': enc,
            'Model Score': round(pred, 3) if pred is not None else None,
            'Hit (>4 VORP)': actual,
            'Sum VORP 4yr': round(vorp, 2) if pd.notna(vorp) else None,
            'Distance': round(dist, 3)
        })

    results_df = pd.DataFrame(results)
    results_df.index = range(1, len(results_df) + 1)
    results_df.index.name = 'Rank'
    pd.set_option('display.max_rows', 20)
    pd.set_option('display.float_format', '{:.3f}'.format)
    display(results_df)
    return results_df


def find_similar_by_manual_input(cluster, raw_inputs, model_score=None, top_n=10, label='Custom Prospect'):
    """
    Find the most similar historical prospects to a hypothetical player.
    Pass model_score (0-1) if known, or leave None to exclude it from similarity.
    """
    model_data = models_by_cluster[cluster]
    features = model_data['features']

    extra_features = [f for f in ['Height_in', 'Player_Encoded'] if f in final_df_transform.columns]
    sim_features = features + [f for f in extra_features if f not in features]

    cluster_df = final_df_transform[final_df_transform['PlayStyleCluster'] == cluster].copy()
    cluster_df[sim_features] = cluster_df[sim_features].fillna(0)

    pred_df = model_data['df_with_predictions'][['Name', 'Prediction']].copy()
    cluster_df = cluster_df.merge(pred_df.rename(columns={'Prediction': '_model_score'}), on='Name', how='left')
    cluster_df['_model_score'] = cluster_df['_model_score'].fillna(0)
    sim_features_with_score = sim_features + ['_model_score']

    sim_scaler = StandardScaler()
    X_scaled = sim_scaler.fit_transform(cluster_df[sim_features_with_score])

    input_vector = {feat: raw_inputs.get(feat, 0.0) for feat in sim_features}
    input_vector['_model_score'] = model_score if model_score is not None else 0.0
    df_input = pd.DataFrame([input_vector])
    target_scaled = sim_scaler.transform(df_input)

    distances = cdist(target_scaled, X_scaled, metric='euclidean')[0]
    cluster_df['_distance'] = distances
    sorted_df = cluster_df.sort_values('_distance')

    pred_lookup = pred_df.set_index('Name')

    cluster_labels = {0.0: 'Centers', 1.0: 'Wings/Forwards', 2.0: 'Guards'}
    print(f"\nMost similar prospects to {label} ({cluster_labels.get(cluster, f'Cluster {cluster}')})")
    print(f"Features used: {sim_features_with_score}\n")

    results = []
    for _, sim_row in sorted_df.head(top_n).iterrows():
        name = sim_row['Name']
        year = sim_row.get('Year', '?')
        dist = sim_row['_distance']
        vorp = sim_row.get('sum_vorp_4yrs', None)
        pick = sim_row.get('Pick', None)
        height = sim_row.get('Height_in', None)
        enc = sim_row.get('Player_Encoded', None)
        pred = pred_lookup.loc[name, 'Prediction'] if name in pred_lookup.index else None
        actual = model_data['df_with_predictions'].set_index('Name').loc[name, 'Actual'] if name in model_data['df_with_predictions']['Name'].values else None
        results.append({
            'Name': name,
            'Year': year,
            'Pick': pick,
            'Height': round(height, 1) if pd.notna(height) else None,
            'Class': enc,
            'Model Score': round(pred, 3) if pred is not None else None,
            'Hit (>4 VORP)': actual,
            'Sum VORP 4yr': round(vorp, 2) if pd.notna(vorp) else None,
            'Distance': round(dist, 3)
        })

    results_df = pd.DataFrame(results)
    results_df.index = range(1, len(results_df) + 1)
    results_df.index.name = 'Rank'
    pd.set_option('display.max_rows', 20)
    pd.set_option('display.float_format', '{:.3f}'.format)
    display(results_df)
    return results_df


In [ ]:
# Example: find historical comps for a player by name
find_similar_players('Karl-Anthony Towns', top_n=10)


In [ ]:
# ─── Manual Prospect Comp Finder ───────────────────────────────────────────
# Set the cluster for your prospect:
#   0.0 = Centers  |  1.0 = Wings/Forwards  |  2.0 = Guards
#
# Player_Encoded: Fr=1, So=2, Jr=3, Sr=4
# model_score: run explain_manual_prediction() first and paste the result here
# ─────────────────────────────────────────────────────────────────────────────

prospect_cluster = 1.0

# Cluster 0.0 (Centers) features: DraftValue, LogDR, DBPM, LogUsg, Player_Encoded
# Cluster 1.0 (Wings)   features: LogUsg/Ast, DraftValue, LogREB, LogFT%, TO, Player_Encoded
# Cluster 2.0 (Guards)  features: LogFT%, DraftValue, close_min, Close 2%, Player_Encoded

prospect_inputs = {
    # ── Model features (fill in for your cluster) ──
    'LogUsg/Ast':     0.7080357930537,   # cluster 1
    'DraftValue':     0.35,   # all clusters
    'LogREB':         3.4011973816622,   # cluster 1
    'LogFT%':         0.53,   # clusters 1, 2
    'TO':             16.7,   # cluster 1
    'LogDR':          3.081909969795,   # cluster 0
    'DBPM':           7.6,   # cluster 0
    'LogUsg':         3.2228678461377,   # cluster 0
    'close_min':      1.232,   # cluster 2
    'Close 2 %':       .65,   # cluster 2
    # ── Extra similarity features ──
    'Height_in':      81,   # height in inches (e.g. 6'6" = 78)
    'Pick':           30,   # draft pick number
    'Player_Encoded': 4,     # Fr=1, So=2, Jr=3, Sr=4
}

prospect_model_score = .77 # paste output from explain_manual_prediction(), e.g. 0.42

find_similar_by_manual_input(
    cluster=prospect_cluster,
    raw_inputs=prospect_inputs,
    model_score=prospect_model_score,
    top_n=10,
    label='My Prospect'
)


In [ ]:
#### Rookie Year 2025 Accuracy Test

In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv("/Users/dannyharlan/Desktop/25 test - Sheet1.csv")


# Extract year from parentheses into a new column before removing it
df["Year"] = df["Name"].str.extract(r"\((\d+)\)")  # Extracts the numeric part inside parentheses

# Now remove the year in parentheses from 'Name' column
df["Name"] = df["Name"].str.replace(r"\s*\(\d+\)", "", regex=True).str.strip()

# Preview the first few rows
print(df.head())

# Optional: Check data types and convert Year to numeric if needed
print("\nData types before conversion:")
print(df.dtypes)

df["Year"] = pd.to_numeric(df["Year"], errors='coerce')  # Convert to numeric, invalid parsing becomes NaN

print("\nData types after conversion:")
print(df.dtypes)

# Remove rows where Year is 8 or 9
df = df[~df['Year'].isin([8, 9])]

# Alternative syntax
df = df[(df['Year'] != 8) & (df['Year'] != 9)]
df.head(10)



In [ ]:
import pandas as pd
import pickle
import numpy as np
from scipy import stats

# Load CSV
csv_df = pd.read_csv("/Users/dannyharlan/Desktop/25 test - Sheet1.csv")
csv_df["Year"] = csv_df["Name"].str.extract(r"\((\d+)\)")
csv_df["Name"] = csv_df["Name"].str.replace(r"\s*\(\d+\)", "", regex=True).str.strip()
csv_df["Year"] = pd.to_numeric(csv_df["Year"], errors='coerce')
csv_df = csv_df[~csv_df['Year'].isin([8, 9])].copy()

# Pull model predictions
with open('models_by_cluster.pkl', 'rb') as f:
  models = pickle.load(f)

all_preds = pd.concat([data['df_with_predictions'][['Name','Year','Prediction']] for data in models.values()])
df = csv_df.merge(all_preds, on=['Name','Year'], how='left').rename(columns={'Prediction': 'Model_Rating'})
df = df.dropna(subset=['VORP YR1', 'Model_Rating'])

# Spearman rank correlations
model_corr, model_p = stats.spearmanr(df['Model_Rating'], df['VORP YR1'])
pick_corr, pick_p   = stats.spearmanr(-df['Pick'], df['VORP YR1'])  # negate: lower pick = better

print("=== Model Rating vs VORP YR1 ===")
print(f"  Spearman r = {model_corr:.3f}  (p={model_p:.3f})")
print()
print("=== Draft Pick vs VORP YR1 ===")
print(f"  Spearman r = {pick_corr:.3f}  (p={pick_p:.3f})")
print()

# Table sorted by model rating
out = df[['Name', 'Year', 'Pick', 'Model_Rating', 'VORP YR1']].sort_values('Model_Rating', ascending=False).copy()
out['Model_Rating'] = out['Model_Rating'].round(3)
print(out.to_string(index=False))


In [ ]:
df['Model_Rank'] = df['Model_Rating'].rank(ascending=False)
df['Pick_Rank']  = df['Pick'].rank(ascending=True)
df['VORP_Rank']  = df['VORP YR1'].rank(ascending=False)

df['Model_Err'] = (df['Model_Rank'] - df['VORP_Rank']).abs()
df['Pick_Err']  = (df['Pick_Rank']  - df['VORP_Rank']).abs()
df['Model_Edge'] = df['Pick_Err'] - df['Model_Err']  # positive = model was closer

out = df[['Name', 'Pick', 'Model_Rating', 'VORP YR1', 'Model_Rank', 'Pick_Rank', 'VORP_Rank', 'Model_Err', 'Pick_Err', 'Model_Edge']].sort_values('Model_Edge',
ascending=False)
out['Model_Rating'] = out['Model_Rating'].round(3)

print("=== Where model beat the draft (top = biggest model advantage) ===")
print(out.head(30).to_string(index=False))
print()
print("=== Where draft beat the model (top = biggest draft advantage) ===")
print(out.tail(10).to_string(index=False))

In [ ]:
from sklearn.metrics import ndcg_score
import numpy as np

# VORP as relevance scores — higher VORP = more relevant
# Need to handle negatives: shift so min = 0
vorp_relevance = df['VORP YR1'] - df['VORP YR1'].min()

# ndcg_score expects shape (1, n_samples)
# Second arg is the scores your ranker assigned (higher = ranked higher)
model_ndcg = ndcg_score([vorp_relevance.values], [df['Model_Rating'].values])
pick_ndcg  = ndcg_score([vorp_relevance.values], [-df['Pick'].values])  # negate: lower pick = better

print(f"Model NDCG: {model_ndcg:.3f}")
print(f"Draft NDCG: {pick_ndcg:.3f}")

# Also run top-10 only — how well did each find the actual 10 best rookies
top10_vorp = df.nlargest(10, 'VORP YR1')['Name'].tolist()
model_top10 = df.nlargest(10, 'Model_Rating')['Name'].tolist()
draft_top10 = df.nsmallest(10, 'Pick')['Name'].tolist()

model_hits = len(set(model_top10) & set(top10_vorp))
draft_hits = len(set(draft_top10) & set(top10_vorp))
print(f"\nTop-10 overlap with actual top-10 VORP:")
print(f"  Model: {model_hits}/10")
print(f"  Draft: {draft_hits}/10")


In [ ]:
top10_actual = df.nlargest(45, 'VORP YR1')[['Name', 'VORP YR1']].reset_index(drop=True)
top10_model  = df.nlargest(45, 'Model_Rating')[['Name', 'Model_Rating']].reset_index(drop=True)
top10_draft  = df.nsmallest(45, 'Pick')[['Name', 'Pick']].reset_index(drop=True)

comparison = pd.DataFrame({
  'Actual Top 10 (VORP)': top10_actual['Name'] + ' (' + top10_actual['VORP YR1'].astype(str) + ')',
  'Model Top 10':         top10_model['Name'] + ' (' + top10_model['Model_Rating'].round(3).astype(str) + ')',
  'Draft Top 10':         top10_draft['Name'] + ' (pick ' + top10_draft['Pick'].astype(str) + ')'
})

print(comparison.to_string(index=False))



In [ ]:
for n in [5, 10, 15]:
  top_n = df.nlargest(n, 'VORP YR1')
  model_corr, _ = stats.spearmanr(top_n['Model_Rating'], top_n['VORP YR1'])
  pick_corr, _  = stats.spearmanr(-top_n['Pick'], top_n['VORP YR1'])
  print(f"Top-{n}:  Model {model_corr:.3f}  |  Draft {pick_corr:.3f}")

In [ ]:

for threshold in [0.1, 0.5, 0.75]:
  top_n = df[df['VORP YR1'] >= threshold]
  n = len(top_n)
  model_corr, _ = stats.spearmanr(top_n['Model_Rating'], top_n['VORP YR1'])
  pick_corr, _  = stats.spearmanr(-top_n['Pick'], top_n['VORP YR1'])
  print(f"VORP >= {threshold} (n={n}):  Model {model_corr:.3f}  |  Draft {pick_corr:.3f}")

In [ ]:
lottery = df[df['Pick'] <= 14]
model_corr, model_p = stats.spearmanr(lottery['Model_Rating'], lottery['VORP YR1'])
pick_corr, pick_p   = stats.spearmanr(-lottery['Pick'], lottery['VORP YR1'])
print(f"Lottery (picks 1-14):  Model {model_corr:.3f} (p={model_p:.3f})  |  Draft {pick_corr:.3f} (p={pick_corr:.3f})")


In [ ]:
high_rated = df[df['Model_Rating'] > 0.75]
n = len(high_rated)
model_corr, model_p = stats.spearmanr(high_rated['Model_Rating'], high_rated['VORP YR1'])
pick_corr, pick_p   = stats.spearmanr(-high_rated['Pick'], high_rated['VORP YR1'])
print(f"Model > 0.75 (n={n}):  Model {model_corr:.3f} (p={model_p:.3f})  |  Draft {pick_corr:.3f} (p={pick_p:.3f})")

In [ ]:
vorp_filtered = df[df['VORP YR1'] >= 0.1]
n = len(vorp_filtered)
model_corr, model_p = stats.spearmanr(vorp_filtered['Model_Rating'], vorp_filtered['VORP YR1'])
pick_corr, pick_p   = stats.spearmanr(-vorp_filtered['Pick'], vorp_filtered['VORP YR1'])
print(f"VORP >= 0.1 (n={n}):  Model {model_corr:.3f} (p={model_p:.3f})  |  Draft {pick_corr:.3f} (p={pick_p:.3f})")

In [ ]:
clear_signal = df[(df['VORP YR1'] >= 0.1) | (df['VORP YR1'] < -0.1)]
n = len(clear_signal)
model_corr, model_p = stats.spearmanr(clear_signal['Model_Rating'], clear_signal['VORP YR1'])
pick_corr, pick_p   = stats.spearmanr(-clear_signal['Pick'], clear_signal['VORP YR1'])
print(f"Clear signal (n={n}):  Model {model_corr:.3f} (p={model_p:.3f})  |  Draft {pick_corr:.3f} (p={pick_p:.3f})")


In [ ]:
mpg_df = pd.read_csv("/Users/dannyharlan/Desktop/ddd - Sheet1.csv")
print(mpg_df.head())
print(mpg_df.columns.tolist())

In [ ]:
mpg_df = pd.read_csv("/Users/dannyharlan/Desktop/ddd - Sheet1.csv")
df = df.merge(mpg_df[['Name', 'MP', 'G']], on='Name', how='left')

print(df[['Name', 'Pick', 'VORP YR1', 'MP']].sort_values('MP', ascending=False).to_string(index=False))

In [ ]:
print(mpg_df['Name'].tolist())

In [ ]:
name_map = {
  'V.J. Edgecombe': 'VJ Edgecombe',
  'Walter Clayton Jr.': 'Walter Clayton',
  'Egor Demin': 'Egor Dёmin',
  'Kasparas Jakucionis': 'Kasparas Jakučionis',
  'Yanic Konan Niederhauser': 'Yanic Konan Niederhäuser',
}

df['Name_merge'] = df['Name'].replace(name_map)
mpg_df['Name_merge'] = mpg_df['Name']

df = df.drop(columns=['MP', 'G'], errors='ignore')
df = df.merge(mpg_df[['Name_merge', 'MP', 'G']], on='Name_merge', how='left').drop(columns='Name_merge')

print(df[['Name', 'Pick', 'VORP YR1', 'MP']].sort_values('MP', ascending=False).head(100).to_string(index=False))

In [ ]:
played = df[df['MP'] >= 500]
n = len(played)
model_corr, model_p = stats.spearmanr(played['Model_Rating'], played['VORP YR1'])
pick_corr, pick_p   = stats.spearmanr(-played['Pick'], played['VORP YR1'])
print(f"MP >= 500 (n={n}):  Model {model_corr:.3f} (p={model_p:.3f})  |  Draft {pick_corr:.3f} (p={pick_p:.3f})")

In [ ]:
from scipy import stats as scipy_stats

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Model Rating vs VORP
axes[0].scatter(played['Model_Rating'], played['VORP YR1'], color='steelblue', edgecolors='black', alpha=0.7)
for _, row in played.iterrows():
  axes[0].annotate(row['Name'].split()[-1], (row['Model_Rating'], row['VORP YR1']), fontsize=7, alpha=0.7)
m, b, _, _, _ = scipy_stats.linregress(played['Model_Rating'], played['VORP YR1'])
x = np.linspace(played['Model_Rating'].min(), played['Model_Rating'].max(), 100)
axes[0].plot(x, m*x + b, color='navy', linewidth=2)
axes[0].set_xlabel('Model Rating')
axes[0].set_ylabel('VORP YR1')
axes[0].set_title(f'Model Rating vs VORP YR1 (r={model_corr:.3f}, p={model_p:.3f})')

# Draft Pick vs VORP
axes[1].scatter(-played['Pick'], played['VORP YR1'], color='tomato', edgecolors='black', alpha=0.7)
for _, row in played.iterrows():
  axes[1].annotate(row['Name'].split()[-1], (-row['Pick'], row['VORP YR1']), fontsize=7, alpha=0.7)
m2, b2, _, _, _ = scipy_stats.linregress(-played['Pick'], played['VORP YR1'])
x2 = np.linspace(-played['Pick'].max(), -played['Pick'].min(), 100)
axes[1].plot(x2, m2*x2 + b2, color='darkred', linewidth=2)
axes[1].set_xlabel('Draft Pick (negated)')
axes[1].set_ylabel('VORP YR1')
axes[1].set_title(f'Draft Pick vs VORP YR1 (r={pick_corr:.3f}, p={pick_p:.3f})')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import numpy as np

# Get training data predictions and actuals
train_preds, train_actuals = [], []
for cluster, data in models_by_cluster.items():
  df = data['df_with_predictions']
  train = df[df['Year'] <= 18]
  train_preds.extend(train['Prediction'].tolist())
  train_actuals.extend(train['Actual'].tolist())

train_preds = np.array(train_preds)
train_actuals = np.array(train_actuals)

# AUC-ROC
auc = roc_auc_score(train_actuals, train_preds)
print(f"AUC-ROC: {auc:.3f}")

# Side by side plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(train_actuals, train_preds)
axes[0].plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve (Training 2010-2018)')
axes[0].legend()

# Calibration Curve
prob_true, prob_pred = calibration_curve(train_actuals, train_preds, n_bins=8)
axes[1].plot(prob_pred, prob_true, marker='o', color='steelblue', linewidth=2, label='Model')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Fraction Actually Successful')
axes[1].set_title('Calibration Curve (Training 2010-2018)')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_fscore_support, classification_report

# Try a few thresholds
for threshold in [0.85]:
  preds_binary = (train_preds >= threshold).astype(int)
  p, r, f1, _ = precision_recall_fscore_support(train_actuals, preds_binary, average='binary', zero_division=0)
  print(f"Threshold {threshold}: Precision={p:.3f}  Recall={r:.3f}  F1={f1:.3f}")

print()
print("Full report at 0.85:")
print(classification_report(train_actuals, (train_preds >= 0.85).astype(int), zero_division=0))

In [ ]:
from sklearn.metrics import matthews_corrcoef

for threshold in [0.7]:
  preds_binary = (train_preds >= threshold).astype(int)
  mcc = matthews_corrcoef(train_actuals, preds_binary)
  print(f"Threshold {threshold}: MCC = {mcc:.3f}")


In [ ]:
import pandas as pd
import numpy as np
import re
from scipy.stats import spearmanr
from nba_api.stats.endpoints import drafthistory

# --- load DARKO and draft ---
darko = pd.read_csv('DARKO - Daily Adjusted and Regressed Kalman Optimized projections - Full DPM History.csv')
darko = darko.groupby(['nba_id', 'season']).agg(dpm=('dpm', 'mean')).reset_index()
darko_idx = darko.set_index(['nba_id', 'season'])['dpm']

draft = drafthistory.DraftHistory().get_data_frames()[0]
draft = draft[['PERSON_ID', 'PLAYER_NAME', 'SEASON', 'OVERALL_PICK']].rename(columns={'PERSON_ID': 'nba_id', 'SEASON': 'draft_year'})
draft['draft_year'] = draft['draft_year'].astype(str).str[:4].astype(int)

def norm_name(s):
    s = str(s).lower()
    s = re.sub(r'\b(jr|sr|ii|iii|iv)\b\.?', '', s)
    s = s.replace('.', '').replace('-', ' ').replace("'", '')
    return re.sub(r'\s+', ' ', s).strip()

draft['name_key'] = draft['PLAYER_NAME'].apply(norm_name)

# --- gather all predictions ---
all_preds = pd.concat([
    v['df_with_predictions'][['Name', 'Year', 'Prediction']]
    for v in models_by_cluster.values()
], ignore_index=True).drop_duplicates(subset=['Name', 'Year'])

all_preds['name_key'] = all_preds['Name'].apply(norm_name)
all_preds['draft_year'] = all_preds['Year'].astype(int) + 2000

# --- match to NBA by name + year ---
matched = all_preds.merge(
    draft[['nba_id', 'name_key', 'draft_year', 'OVERALL_PICK']],
    on=['name_key', 'draft_year'], how='left'
)
# try year+1 for unmatched
unmatched = matched[matched['nba_id'].isna()].copy()
unmatched['draft_year'] = unmatched['draft_year'] + 1
unmatched = unmatched.drop(columns=['nba_id', 'OVERALL_PICK']).merge(
    draft[['nba_id', 'name_key', 'draft_year', 'OVERALL_PICK']],
    on=['name_key', 'draft_year'], how='left'
)
matched = pd.concat([matched[matched['nba_id'].notna()], unmatched[unmatched['nba_id'].notna()]], ignore_index=True)

# --- get Y5 DARKO DPM ---
def get_y5(row):
    try:
        return darko_idx.loc[(row['nba_id'], row['draft_year'] + 5)]
    except:
        return np.nan

matched['nba_dpm_y5'] = matched.apply(get_y5, axis=1)

# --- evaluate ---
eval_df = matched.dropna(subset=['nba_dpm_y5', 'Prediction']).copy()
r, p = spearmanr(eval_df['Prediction'], eval_df['nba_dpm_y5'])
print(f'Matched: {len(matched)} | With Y5 DPM: {len(eval_df)}')
print(f'Spearman vs Y5 DPM: {r:.3f} (p={p:.3f})')
print()
print(eval_df.sort_values('Prediction', ascending=False)
      [['Name', 'draft_year', 'OVERALL_PICK', 'Prediction', 'nba_dpm_y5']]
      .head(50).to_string(index=False))
